# Mid-Cap Stock Selection Factor Model | Bloomberg Terminal & Python


<a id="section-1"></a>

## 1. Project Overview

**Research question:** Can a point-in-time factor model emphasizing accelerating fundamentals, business quality, and growth-adjusted valuation identify U.S. mid-cap stocks that subsequently outperform comparable mid-cap equities over a 36-month holding period?

This project uses Bloomberg Terminal point-in-time data and Python to construct and historically test a systematic mid-cap stock-selection model. Bloomberg historical screening and Excel/BQL-style data workflows support historical universe reconstruction, currency normalization, economic-company deduplication, and historical GICS normalization. Python, pandas, and NumPy turn the historical inputs into cross-sectional factor rankings and portfolio backtests.

The composite model combines **Growth (50%)**, **Quality (30%)**, and **Valuation (20%)**. Growth contains Revenue Growth Acceleration and 3-month BF12M EPS Revision; Quality contains ROIC and FCF Margin; Valuation contains BF12M PEG and Growth-Adjusted EV / Forward Sales. Accelerating fundamentals receive the largest emphasis, while business quality and growth-adjusted valuation also contribute to every company's final score.

Each historical screen forms a portfolio of the **Top 25 factor-ranked companies**. The primary outcome is **36-month cumulative total return**, evaluated against **100 random 25-stock portfolios drawn from the same eligible scoring universe**, the **S&P MidCap 400 (MID Index)**, and the **Bottom 25 factor-ranked companies**. Annualized volatility and maximum drawdown provide risk context, and an empirical random-portfolio test evaluates how unusual the Top 25 return is among sampled peer portfolios. CAGR is reported separately as a supplementary annualized measure.

An exploratory secondary analysis of large-cap graduation appears in the appendix.

### Contents

1. [Project Overview](#section-1)
2. [Imports & Configuration](#section-2)
3. [Historical Cohort Data](#section-3)
4. [Universe Construction](#section-4)
5. [Data Cleaning & Economic-Company Deduplication](#section-5)
6. [Factor Construction](#section-6)
7. [Historical Sector Normalization](#section-7)
8. [Factor Transformation & Scoring](#section-8)
9. [Portfolio Formation](#section-9)
10. [36-Month Return Analysis](#section-10)
11. [Random Portfolio Benchmark](#section-11)
12. [S&P MidCap 400 Benchmark](#section-12)
13. [Risk Analysis](#section-13)
14. [Return Statistical Testing](#section-14)
15. [Final Results](#section-15)
16. [Final Project Charts](#section-16)
17. [Conclusion](#section-17)
18. [Appendix — Exploratory Large-Cap Graduation Analysis](#section-18)

<a id="section-2"></a>

## 2. Imports & Configuration

Run the notebook from top to bottom with Python, Jupyter/IPython, and the following packages: `pandas`, `numpy`, `scipy`, `openpyxl`, `yfinance`, `jinja2`, and `matplotlib`. The recorded Python version is 3.14.6; package versions are not pinned.

File paths are relative to the working directory. Run with the private research data folder as the working directory. The main analysis requires the three universe workbooks and benchmark workbook. The appendix additionally requires the identity map, `Snapshots/` directory, and network access for historical FX.

**Data access:** Bloomberg source workbooks are proprietary and excluded from the public repository. Historical screen-date FX constants are specified in the notebook. Public outputs show aggregate research results. Local execution can generate security-level data previews and a detailed Excel export; exclude those outputs and workbooks from public commits.

In [ ]:
import re
from pathlib import Path
from difflib import SequenceMatcher
from itertools import combinations

import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import fisher_exact
from IPython.display import display

files = {
    2014: Path("FINAL_US_UNIVERSE_2014.xlsx"),
    2019: Path("FINAL_US_UNIVERSE_2019.xlsx"),
    2022: Path("FINAL_US_UNIVERSE_2022.xlsx"),
}

benchmark_file = Path("Benchmark_Monthly_Returns.xlsx")
ticker_map_file = Path("Ticker_ID_Map_Working.xlsx")
snapshot_dir = Path("Snapshots")


# Required private security-level research inputs; intentionally excluded from
# the public repository. Place private_overrides.json in the private research
# working directory. Complete local execution requires the original values.
# Preserve values and list order to reproduce the finalized research.
# Required JSON structure (types only; no sample security values):
# manual_alias_groups: year string -> list of {"aliases": list[str], "preferred": str}
# gics_map_2014, missing_2019_map: full security identifier -> historical sector
# pre_2023_sector_map: base ticker -> historical sector
# alias_groups: economic-company key -> list of full security identifiers
# cross_listing_check: {"foreign": str, "us": str} (foreign minus us; denominator us)
# snapshot_special_economic_key: str (original synthetic company key)
# snapshot_preferred_records: full security identifier -> bool
import json

private_overrides_file = Path("private_overrides.json")


def _private_config_error(detail):
    return ValueError(
        "Invalid private_overrides.json: " + detail
        + " Supply the complete original security-level research inputs. "
        "This private file is intentionally excluded from the public repository."
    )


def _private_json_object(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise _private_config_error("duplicate object keys are not permitted.")
        result[key] = value
    return result


try:
    private_overrides = json.loads(
        private_overrides_file.read_text(encoding="utf-8"),
        object_pairs_hook=_private_json_object,
    )
except FileNotFoundError:
    raise FileNotFoundError(
        "Required private_overrides.json was not found in the working directory. "
        "Complete local execution requires the original private security-level "
        "research mappings and overrides. This file is intentionally excluded "
        "from the public repository. Supply it alongside the private research "
        "inputs before running this notebook; no empty or sample mappings are used."
    ) from None
except (json.JSONDecodeError, UnicodeError):
    raise _private_config_error("the file must contain valid UTF-8 JSON.") from None
except OSError:
    raise _private_config_error("the private file could not be read.") from None


def _private_text(value):
    return isinstance(value, str) and bool(value.strip())


def _private_text_list(value):
    return isinstance(value, list) and bool(value) and all(map(_private_text, value))


_required_private_keys = {
    "manual_alias_groups", "gics_map_2014", "pre_2023_sector_map",
    "missing_2019_map", "alias_groups", "cross_listing_check",
    "snapshot_special_economic_key", "snapshot_preferred_records",
}
if not isinstance(private_overrides, dict) or not _required_private_keys.issubset(private_overrides):
    raise _private_config_error("all documented top-level keys are required.")
for _name in ("gics_map_2014", "pre_2023_sector_map", "missing_2019_map"):
    _mapping = private_overrides[_name]
    if not isinstance(_mapping, dict) or not _mapping or not all(
        _private_text(k) and _private_text(v) for k, v in _mapping.items()
    ):
        raise _private_config_error(_name + " must be a nonempty string-to-string object.")
_groups = private_overrides["manual_alias_groups"]
if not isinstance(_groups, dict) or set(_groups) != {"2014", "2019", "2022"}:
    raise _private_config_error("manual_alias_groups requires the three cohort year keys.")
for _year_groups in _groups.values():
    if not isinstance(_year_groups, list) or not _year_groups:
        raise _private_config_error("each cohort requires its original alias groups.")
    for _group in _year_groups:
        if (not isinstance(_group, dict) or set(_group) != {"aliases", "preferred"}
                or not _private_text_list(_group["aliases"])
                or not _private_text(_group["preferred"])
                or _group["preferred"] not in _group["aliases"]):
            raise _private_config_error("manual alias groups require aliases and a preferred member.")
_groups = private_overrides["alias_groups"]
if not isinstance(_groups, dict) or not _groups or not all(
    _private_text(k) and _private_text_list(v) for k, v in _groups.items()
):
    raise _private_config_error("alias_groups requires company keys and nonempty alias lists.")
_special_key = private_overrides["snapshot_special_economic_key"]
if not _private_text(_special_key) or _special_key not in _groups:
    raise _private_config_error("snapshot_special_economic_key must identify an alias group.")
_records = private_overrides["snapshot_preferred_records"]
if not isinstance(_records, dict) or not _records or not all(
    _private_text(k) and k in _groups[_special_key] and isinstance(v, bool)
    for k, v in _records.items()
):
    raise _private_config_error("snapshot_preferred_records requires special-group identifiers and booleans.")
_check = private_overrides["cross_listing_check"]
if (not isinstance(_check, dict) or set(_check) != {"foreign", "us"}
        or not all(map(_private_text, _check.values())) or _check["foreign"] == _check["us"]):
    raise _private_config_error("cross_listing_check requires distinct foreign and us identifiers.")



<a id="section-3"></a>

## 3. Historical Cohort Data

The cohort screen dates are **2014-12-31**, **2019-12-31**, and **2022-12-30**. Their 36-month evaluation windows are January 2015–December 2017, January 2020–December 2022, and January 2023–December 2025.

Private inputs comprise:
- `FINAL_US_UNIVERSE_2014.xlsx`, `FINAL_US_UNIVERSE_2019.xlsx`, and `FINAL_US_UNIVERSE_2022.xlsx`: `Sheet1` contains company characteristics; `Monthly Returns` contains subsequent returns.
- `Benchmark_Monthly_Returns.xlsx`: one S&P MidCap 400 return sheet per cohort.

The loading checks assess field coverage before standardizing inconsistent export column names.

In [ ]:
final_universes = {}

for year, file in files.items():

    df = pd.read_excel(
        file,
        sheet_name="Sheet1",
        header=2
    )

    # Remove Bloomberg's "None (3000 securities)" row
    df = df[
        df["Ticker"].astype(str).str.contains("Equity", na=False)
    ].copy()

    df = df.reset_index(drop=True)

    final_universes[year] = df

    print(f"\n{'=' * 70}")
    print(year)
    print(f"{'=' * 70}")
    print("Rows:", len(df))
    print("Unique tickers:", df["Ticker"].nunique())

    print("\nColumns:")
    for col in df.columns:
        print(" -", col)


In [ ]:
# --------------------------------------------------
# Audit required Bloomberg fields and coverage
# --------------------------------------------------

required_columns = [
    "Ticker",
    "Short Name",
    "Market Cap",
    "BB Company ID",
    "GICS Sector",
    "3M ADTV",
    "ROIC",
    "FCF MARGIN",
    "Revenue 0AQ",
    "Revenue -1Q",
    "Revenue -4Q",
    "Revenue -5Q",
    "BF12M EPS",
    "BF12M EPS 3M AGO",
    "Best PEG Ratio BF12M",
    "Next-Year Estimated Sales",
    "2FY Estimated Sales",
    "historical Enterprise Value",
    "Pricing Currency",
    "Market Cap USD"
]

for year, df in final_universes.items():

    print(f"\n{'=' * 70}")
    print(year)
    print(f"{'=' * 70}")

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    print("Rows:", len(df))
    print("Missing required columns:", missing_columns if missing_columns else "None")

    print("\nPopulated values:")

    for col in required_columns:
        if col in df.columns:
            print(
                f"{col:32s}",
                f"{df[col].notna().sum():4d} / {len(df)}"
            )


In [ ]:
# --------------------------------------------------
# Standardize column names across 2014 / 2019 / 2022
# --------------------------------------------------

rename_maps = {
    2014: {
        "BEst PEG Ratio BF12M": "BF12M PEG",
    },

    2019: {
        "Revenue -1AQ": "Revenue -1Q",
        "Revenue -4AQ": "Revenue -4Q",
        "Revenue -5AQ": "Revenue -5Q",
    },

    2022: {
        "Revenue -1AQ": "Revenue -1Q",
        "Revenue -4AQ": "Revenue -4Q",
        "Revenue -5AQ": "Revenue -5Q",
        "Next-Year Estiamted Sales": "Next-Year Estimated Sales",
    }
}

for year in [2014, 2019, 2022]:

    final_universes[year] = final_universes[year].rename(
        columns=rename_maps[year]
    )

required = [
    "Ticker",
    "Short Name",
    "Market Cap",
    "BB Company ID",
    "GICS Sector",
    "3M ADTV",
    "ROIC",
    "FCF MARGIN",
    "Revenue 0AQ",
    "Revenue -1Q",
    "Revenue -4Q",
    "Revenue -5Q",
    "BF12M EPS",
    "BF12M EPS 3M AGO",
    "BF12M PEG",
    "Next-Year Estimated Sales",
    "2FY Estimated Sales",
    "historical Enterprise Value",
    "Pricing Currency"
]

for year, df in final_universes.items():

    missing = [
        col for col in required
        if col not in df.columns
    ]

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    print("Rows:", len(df))
    print(
        "Missing required columns:",
        missing if missing else "None"
    )

    print("\nPricing currencies:")
    print(
        df["Pricing Currency"]
        .value_counts(dropna=False)
        .head(20)
    )


In [ ]:
# --------------------------------------------------
# List every non-USD pricing currency we need
# historical FX rates for
# --------------------------------------------------

all_non_usd_currencies = sorted(
    set().union(
        *[
            set(
                df.loc[
                    df["Pricing Currency"] != "USD",
                    "Pricing Currency"
                ].dropna()
            )
            for df in final_universes.values()
        ]
    )
)

print("Non-USD currencies needed:")
print(all_non_usd_currencies)

print("\nNumber of currencies:", len(all_non_usd_currencies))

for year, df in final_universes.items():

    counts = (
        df.loc[
            df["Pricing Currency"] != "USD",
            "Pricing Currency"
        ]
        .value_counts()
        .sort_index()
    )

    print(f"\n{year}:")
    print(counts.to_dict())


<a id="section-4"></a>

## 4. Universe Construction

The mid-cap universe comprises economic-company market-cap ranks **501–1500**, subject to **3-month average daily trading value above $5 million**. Companies are not reranked after the liquidity filter.

Market capitalization is first converted to USD using screen-date FX rates. Company deduplication follows in Section 5, where the rank band and liquidity filter are applied.


In [ ]:
# --------------------------------------------------
# Convert local-currency market cap to USD
# using historical screen-date FX rates
# --------------------------------------------------

fx_usd_per_unit = {
    2014: {
        "USD": 1.0,
        "AUD": 0.8181239361167368,
        "CAD": 0.8627373933348433,
        "EUR": 1.2110232823476486,
        "GBp": 1.5586418347008133 / 100,
        "HKD": 0.12896542822467777,
        "JPY": 0.008349721943572164,
        "KRW": 0.0009157321379035656,
        "NOK": 0.13385565571098715,
        "NZD": 0.7804878113587139,
        "PLN": 0.2827612121332708,
        "SEK": 0.12812601252801378,
        "TWD": 0.03163450989955426,
    },

    2019: {
        "USD": 1.0,
        "AUD": 0.7029520119169443,
        "CAD": 0.7714802783696958,
        "EUR": 1.1227007517929324,
        "GBp": 1.3267515950593538 / 100,
        "HKD": 0.12838611826998983,
        "KRW": 0.0008661444281243449,
        "NOK": 0.11386123904954687,
        "PLN": 0.26395732657863313,
        "SEK": 0.10705156312966968,
        "TWD": 0.03339584526982031,
    },

    2022: {
        "USD": 1.0,
        "AUD": 0.6804983565443623,
        "CAD": 0.7389873828544646,
        "EUR": 1.0694046150759209,
        "GBp": 1.2077147633533385 / 100,
        "HKD": 0.12817826092731477,
        "ILs": 0.2837828364860866,
        "KRW": 0.0007943626653740151,
        "NOK": 0.10159507198192996,
        "PLN": 0.22854256312743404,
        "SEK": 0.09596689298355603,
        "TWD": 0.03253074447829124,
    }
}

for year, df in final_universes.items():

    df = df.copy()

    # Make sure market cap is numeric
    df["Market Cap"] = pd.to_numeric(
        df["Market Cap"],
        errors="coerce"
    )

    # Map historical FX rate
    df["FX_to_USD"] = df["Pricing Currency"].map(
        fx_usd_per_unit[year]
    )

    # Convert to USD
    df["Market Cap USD"] = (
        df["Market Cap"] * df["FX_to_USD"]
    )

    final_universes[year] = df

    print(f"\n{year}")
    print("-" * 45)
    print(
        "Missing FX rates:",
        df["FX_to_USD"].isna().sum()
    )
    print(
        "Missing USD market caps:",
        df["Market Cap USD"].isna().sum()
    )

    print("\nLargest non-USD-priced companies after conversion:")

    display(
        df.loc[
            df["Pricing Currency"] != "USD",
            [
                "Ticker",
                "Short Name",
                "Pricing Currency",
                "Market Cap",
                "FX_to_USD",
                "Market Cap USD"
            ]
        ]
        .sort_values(
            "Market Cap USD",
            ascending=False
        )
        .head(10)
    )


<a id="section-5"></a>

## 5. Data Cleaning & Economic-Company Deduplication

Multiple securities can represent the same economic company. Records with identical USD market capitalization are grouped, with the canonical record selected by factor coverage, non-legacy ticker status, and raw rank. Confirmed alias groups address additional duplicates; available values from those records fill missing fields.

Duplicate-size and near-duplicate audits support the identity decisions. Ticker aliases remain available for return matching. The resulting economic companies are ranked before applying the market-cap band and liquidity filter.


In [ ]:
# --------------------------------------------------
# Find repeated USD market-cap groups near the
# company-ranking range we care about
# --------------------------------------------------

duplicate_groups = {}

for year, df in final_universes.items():

    work = df.copy()

    # Sort by corrected USD market cap
    work = work.sort_values(
        "Market Cap USD",
        ascending=False
    ).reset_index(drop=True)

    work["Raw_USD_Rank"] = range(1, len(work) + 1)

    # Give ourselves a buffer below rank 1500
    check = work[
        work["Raw_USD_Rank"] <= 1700
    ].copy()

    # Exact repeated market caps
    counts = check["Market Cap USD"].value_counts()

    repeated_caps = counts[counts > 1].index

    candidates = check[
        check["Market Cap USD"].isin(repeated_caps)
    ].copy()

    duplicate_groups[year] = candidates

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    print(
        "Rows in repeated-cap groups:",
        len(candidates)
    )

    print(
        "Unique repeated-cap groups:",
        candidates["Market Cap USD"].nunique()
    )

    display(
        candidates[
            [
                "Raw_USD_Rank",
                "Ticker",
                "Short Name",
                "BB Company ID",
                "Pricing Currency",
                "Market Cap USD"
            ]
        ].head(50)
    )


In [ ]:
# --------------------------------------------------
# Inspect duplicate-group sizes before collapsing
# --------------------------------------------------

duplicate_group_summary = {}

for year, candidates in duplicate_groups.items():

    summary = (
        candidates
        .groupby("Market Cap USD", as_index=False)
        .agg(
            Group_Size=("Ticker", "size"),
            Min_Rank=("Raw_USD_Rank", "min"),
            Max_Rank=("Raw_USD_Rank", "max"),
            Tickers=("Ticker", lambda x: " | ".join(x.astype(str))),
            Names=("Short Name", lambda x: " | ".join(x.astype(str)))
        )
        .sort_values("Min_Rank")
        .reset_index(drop=True)
    )

    duplicate_group_summary[year] = summary

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    print("Group-size distribution:")
    print(summary["Group_Size"].value_counts().sort_index())

    unusual = summary[
        summary["Group_Size"] != 2
    ]

    print("\nGroups with size other than 2:")
    if len(unusual) == 0:
        print("None")
    else:
        display(unusual)


In [ ]:
# --------------------------------------------------
# Collapse exact same-market-cap records into
# preliminary economic-company records
#
# Canonical row = row with greatest factor coverage.
# All ticker aliases are preserved for later
# returns / snapshot matching.
# --------------------------------------------------

factor_cols = [
    "ROIC",
    "FCF MARGIN",
    "Revenue 0AQ",
    "Revenue -1Q",
    "Revenue -4Q",
    "Revenue -5Q",
    "BF12M EPS",
    "BF12M EPS 3M AGO",
    "BF12M PEG",
    "Next-Year Estimated Sales",
    "2FY Estimated Sales",
    "historical Enterprise Value"
]

company_universes_exact = {}

for year, df in final_universes.items():

    work = df.copy()

    # Numeric cleanup
    numeric_cols = [
        "Market Cap USD",
        "3M ADTV"
    ] + factor_cols

    for col in numeric_cols:
        work[col] = pd.to_numeric(
            work[col],
            errors="coerce"
        )

    # Rank raw security records by corrected USD market cap
    work = (
        work
        .sort_values(
            "Market Cap USD",
            ascending=False
        )
        .reset_index(drop=True)
    )

    work["Raw_USD_Rank"] = (
        np.arange(1, len(work) + 1)
    )

    # Number of populated raw factor inputs
    work["Factor_Coverage"] = (
        work[factor_cols]
        .notna()
        .sum(axis=1)
    )

    company_rows = []

    for market_cap, group in work.groupby(
        "Market Cap USD",
        sort=False
    ):

        group = group.copy()

        # Bloomberg numeric-D legacy ticker indicator
        group["Legacy_Ticker"] = (
            group["Ticker"]
            .astype(str)
            .str.match(
                r"^\d+D\s+US\s+Equity$",
                na=False
            )
        )

        # Prefer:
        # 1. best factor coverage
        # 2. non-legacy ticker
        # 3. earliest raw rank
        group = group.sort_values(
            [
                "Factor_Coverage",
                "Legacy_Ticker",
                "Raw_USD_Rank"
            ],
            ascending=[
                False,
                True,
                True
            ]
        )

        chosen = group.iloc[0].copy()

        chosen["Alias_Count"] = len(group)

        chosen["Economic_Aliases"] = (
            " | ".join(
                group["Ticker"]
                .astype(str)
                .tolist()
            )
        )

        chosen["Min_Raw_USD_Rank"] = int(
            group["Raw_USD_Rank"].min()
        )

        chosen["Max_Raw_USD_Rank"] = int(
            group["Raw_USD_Rank"].max()
        )

        company_rows.append(chosen)

    company_df = pd.DataFrame(
        company_rows
    )

    company_df = (
        company_df
        .sort_values(
            "Market Cap USD",
            ascending=False
        )
        .reset_index(drop=True)
    )

    company_df["Preliminary_Company_Rank"] = (
        np.arange(1, len(company_df) + 1)
    )

    company_universes_exact[year] = company_df

    rank_500 = company_df.iloc[499]
    rank_1500 = company_df.iloc[1499]

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    print("Raw security rows:", len(work))
    print("Preliminary companies:", len(company_df))
    print(
        "Rows collapsed:",
        len(work) - len(company_df)
    )
    print(
        "Multi-alias companies:",
        (company_df["Alias_Count"] > 1).sum()
    )

    print(
        "\n500th company market cap:",
        f"${rank_500['Market Cap USD']:,.0f}"
    )

    print(
        "1500th company market cap:",
        f"${rank_1500['Market Cap USD']:,.0f}"
    )

    print(
        "Raw rank where 1500th unique company first appears:",
        int(rank_1500["Min_Raw_USD_Rank"])
    )


In [ ]:
# --------------------------------------------------
# Find likely cross-listing / near-duplicate companies
# not caught by exact market-cap matching
# --------------------------------------------------


def clean_company_name(name):
    name = str(name).upper()

    # Remove punctuation
    name = re.sub(r"[^A-Z0-9 ]", " ", name)

    # Remove common corporate suffixes / filler words
    remove_words = [
        "INC", "CORP", "CORPORATION", "CO", "COMPANY",
        "LTD", "LIMITED", "PLC", "LLC", "HOLDINGS",
        "HOLDING", "GROUP", "THE", "SA", "NV", "AG"
    ]

    words = [
        word for word in name.split()
        if word not in remove_words
    ]

    return " ".join(words)


near_duplicate_candidates = {}

for year, df in final_universes.items():

    work = (
        df
        .sort_values("Market Cap USD", ascending=False)
        .head(1700)
        .copy()
        .reset_index(drop=True)
    )

    work["Raw_USD_Rank"] = range(1, len(work) + 1)

    work["Clean_Name"] = (
        work["Short Name"]
        .apply(clean_company_name)
    )

    candidates = []

    for i in range(len(work)):

        row1 = work.iloc[i]

        for j in range(i + 1, len(work)):

            row2 = work.iloc[j]

            cap1 = row1["Market Cap USD"]
            cap2 = row2["Market Cap USD"]

            # Skip if exact same cap — already handled
            if cap1 == cap2:
                continue

            # Only compare reasonably close market caps
            cap_ratio = min(cap1, cap2) / max(cap1, cap2)

            if cap_ratio < 0.95:
                continue

            name_similarity = SequenceMatcher(
                None,
                row1["Clean_Name"],
                row2["Clean_Name"]
            ).ratio()

            if name_similarity >= 0.80:

                candidates.append({
                    "Rank_1": row1["Raw_USD_Rank"],
                    "Ticker_1": row1["Ticker"],
                    "Name_1": row1["Short Name"],
                    "Market_Cap_1": cap1,

                    "Rank_2": row2["Raw_USD_Rank"],
                    "Ticker_2": row2["Ticker"],
                    "Name_2": row2["Short Name"],
                    "Market_Cap_2": cap2,

                    "Cap_Ratio": cap_ratio,
                    "Name_Similarity": name_similarity
                })

    candidates = pd.DataFrame(candidates)

    near_duplicate_candidates[year] = candidates

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    print("Near-duplicate candidates:", len(candidates))

    if len(candidates) > 0:
        display(
            candidates
            .sort_values(
                ["Name_Similarity", "Cap_Ratio"],
                ascending=False
            )
            .head(50)
        )


In [ ]:
# --------------------------------------------------
# Apply confirmed non-exact economic-company merges
# --------------------------------------------------

manual_alias_groups = {int(year): groups for year, groups in private_overrides["manual_alias_groups"].items()}

company_universes_final = {}

for year, df in company_universes_exact.items():

    work = df.copy()

    for merge in manual_alias_groups[year]:

        aliases = merge["aliases"]
        preferred = merge["preferred"]

        # Find any rows containing one of these aliases
        mask = work["Economic_Aliases"].apply(
            lambda x: any(
                alias in str(x).split(" | ")
                for alias in aliases
            )
        )

        matches = work[mask].copy()

        # If exact-cap cleanup already merged them,
        # there may only be one row left.
        if len(matches) <= 1:
            continue

        # Prefer designated primary/current ticker
        preferred_rows = matches[
            matches["Ticker"] == preferred
        ]

        if len(preferred_rows) > 0:
            chosen = preferred_rows.iloc[0].copy()
        else:
            chosen = (
                matches
                .sort_values(
                    "Factor_Coverage",
                    ascending=False
                )
                .iloc[0]
                .copy()
            )

        # Combine aliases from all matched rows
        all_aliases = []

        for alias_string in matches["Economic_Aliases"]:

            for alias in str(alias_string).split(" | "):

                if alias not in all_aliases:
                    all_aliases.append(alias)

        chosen["Economic_Aliases"] = " | ".join(
            all_aliases
        )

        chosen["Alias_Count"] = len(all_aliases)

        # Fill missing factor values from alternate records
        fill_cols = [
            "GICS Sector",
            "3M ADTV",
            "ROIC",
            "FCF MARGIN",
            "Revenue 0AQ",
            "Revenue -1Q",
            "Revenue -4Q",
            "Revenue -5Q",
            "BF12M EPS",
            "BF12M EPS 3M AGO",
            "BF12M PEG",
            "Next-Year Estimated Sales",
            "2FY Estimated Sales",
            "historical Enterprise Value"
        ]

        for col in fill_cols:

            if pd.isna(chosen[col]):

                available = matches[col].dropna()

                if len(available) > 0:
                    chosen[col] = available.iloc[0]

        # Remove old rows
        work = work[~mask].copy()

        # Add combined economic-company record
        work = pd.concat(
            [
                work,
                pd.DataFrame([chosen])
            ],
            ignore_index=True
        )

    # Final company-level ranking
    work = (
        work
        .sort_values(
            "Market Cap USD",
            ascending=False
        )
        .reset_index(drop=True)
    )

    work["Company_Market_Cap_Rank"] = (
        np.arange(1, len(work) + 1)
    )

    company_universes_final[year] = work

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    print("Final economic companies:", len(work))

    print(
        "Companies with multiple aliases:",
        (work["Alias_Count"] > 1).sum()
    )

    print(
        "500th company:",
        work.iloc[499]["Ticker"],
        f"${work.iloc[499]['Market Cap USD']:,.0f}"
    )

    print(
        "1500th company:",
        work.iloc[1499]["Ticker"],
        f"${work.iloc[1499]['Market Cap USD']:,.0f}"
    )


In [ ]:
# --------------------------------------------------
# Build final historical eligible universes
#
# Frozen methodology:
# 1. Company-level market-cap ranks 501–1500
# 2. THEN require 3M ADTV > $5M
# 3. Do NOT rerank after liquidity filter
# --------------------------------------------------

eligible_universes = {}

for year, df in company_universes_final.items():

    work = df.copy()

    # Make sure ADTV is numeric
    work["3M ADTV"] = pd.to_numeric(
        work["3M ADTV"],
        errors="coerce"
    )

    # Fixed market-cap band BEFORE liquidity filtering
    midcap_band = work[
        work["Company_Market_Cap_Rank"].between(
            501,
            1500
        )
    ].copy()

    # Liquidity filter
    eligible = midcap_band[
        midcap_band["3M ADTV"] > 5_000_000
    ].copy()

    eligible_universes[year] = eligible

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    print(
        "Companies ranks 501–1500:",
        len(midcap_band)
    )

    print(
        "Missing ADTV in rank band:",
        midcap_band["3M ADTV"].isna().sum()
    )

    print(
        "Failed $5M ADTV filter:",
        (
            midcap_band["3M ADTV"].notna()
            & (midcap_band["3M ADTV"] <= 5_000_000)
        ).sum()
    )

    print(
        "Final eligible universe:",
        len(eligible)
    )

    print(
        "Rank range retained:",
        int(eligible["Company_Market_Cap_Rank"].min()),
        "to",
        int(eligible["Company_Market_Cap_Rank"].max())
    )

    print(
        "Lowest retained ADTV:",
        f"${eligible['3M ADTV'].min():,.0f}"
    )


<a id="section-6"></a>

## 6. Factor Construction

Revenue growth acceleration is current year-over-year quarterly revenue growth minus the preceding quarter's year-over-year growth. EPS revision is current BF12M EPS divided by its value three months earlier, minus one. Quality uses ROIC and FCF margin directly.

Growth-adjusted EV/sales divides EV/next-year estimated sales by forecast sales growth from next year to year two. Nonpositive forecast sales growth makes this metric missing; nonpositive PEG is also treated as missing. Denominator checks govern whether each derived metric is available.

A company needs at least one available metric in **each** pillar. Two available metrics receive equal weight within a pillar; a single available metric carries the pillar.


In [ ]:
# --------------------------------------------------
# Calculate all derived factor inputs
# --------------------------------------------------

derived_universes = {}

for year, df in eligible_universes.items():

    work = df.copy()

    numeric_cols = [
        "Revenue 0AQ",
        "Revenue -1Q",
        "Revenue -4Q",
        "Revenue -5Q",
        "BF12M EPS",
        "BF12M EPS 3M AGO",
        "BF12M PEG",
        "Next-Year Estimated Sales",
        "2FY Estimated Sales",
        "historical Enterprise Value"
    ]

    for col in numeric_cols:
        work[col] = pd.to_numeric(
            work[col],
            errors="coerce"
        )

    # -----------------------------
    # Revenue growth acceleration
    # -----------------------------

    work["Revenue Current YoY"] = np.where(
        work["Revenue -4Q"].notna()
        & (work["Revenue -4Q"] != 0),

        (work["Revenue 0AQ"] / work["Revenue -4Q"]) - 1,

        np.nan
    )

    work["Revenue Prior YoY"] = np.where(
        work["Revenue -5Q"].notna()
        & (work["Revenue -5Q"] != 0),

        (work["Revenue -1Q"] / work["Revenue -5Q"]) - 1,

        np.nan
    )

    work["Revenue Growth Acceleration"] = (
        work["Revenue Current YoY"]
        - work["Revenue Prior YoY"]
    )

    # -----------------------------
    # 3M BF12M EPS revision
    # Frozen formula:
    # (Current / 3M Ago) - 1
    # -----------------------------

    work["3M BF12M EPS Revision"] = np.where(
        work["BF12M EPS 3M AGO"].notna()
        & (work["BF12M EPS 3M AGO"] != 0),

        (work["BF12M EPS"] / work["BF12M EPS 3M AGO"]) - 1,

        np.nan
    )

    # -----------------------------
    # Valuation inputs
    # -----------------------------

    work["EV/Next Yr Est Sales"] = np.where(
        work["Next-Year Estimated Sales"].notna()
        & (work["Next-Year Estimated Sales"] > 0),

        (
            work["historical Enterprise Value"]
            / work["Next-Year Estimated Sales"]
        ),

        np.nan
    )

    work["BEst Sales YoY Gr:Y+1"] = np.where(
        work["Next-Year Estimated Sales"].notna()
        & (work["Next-Year Estimated Sales"] > 0),

        (
            work["2FY Estimated Sales"]
            / work["Next-Year Estimated Sales"]
        ) - 1,

        np.nan
    )

    # Frozen rule:
    # sales growth <= 0 => Growth-Adjusted EV/Sales missing
    work["Growth-Adjusted EV/Sales"] = np.where(
        work["BEst Sales YoY Gr:Y+1"] > 0,

        (
            work["EV/Next Yr Est Sales"]
            / work["BEst Sales YoY Gr:Y+1"]
        ),

        np.nan
    )

    # Frozen rule:
    # PEG <= 0 => missing
    work["BF12M PEG Clean"] = work["BF12M PEG"].where(
        work["BF12M PEG"] > 0
    )

    derived_universes[year] = work

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    coverage_cols = [
        "Revenue Growth Acceleration",
        "3M BF12M EPS Revision",
        "ROIC",
        "FCF MARGIN",
        "BF12M PEG Clean",
        "Growth-Adjusted EV/Sales"
    ]

    for col in coverage_cols:
        print(
            f"{col:32s}",
            f"{work[col].notna().sum():4d} / {len(work)}"
        )


In [ ]:
# --------------------------------------------------
# Apply frozen pillar-level missing-data rules
# --------------------------------------------------

scoring_universes_pre_gics = {}

for year, df in derived_universes.items():

    work = df.copy()

    # ---------------------------------
    # Pillar availability
    # ---------------------------------

    work["Growth_Available"] = (
        work[
            [
                "Revenue Growth Acceleration",
                "3M BF12M EPS Revision"
            ]
        ]
        .notna()
        .any(axis=1)
    )

    work["Quality_Available"] = (
        work[
            [
                "ROIC",
                "FCF MARGIN"
            ]
        ]
        .notna()
        .any(axis=1)
    )

    work["Valuation_Available"] = (
        work[
            [
                "BF12M PEG Clean",
                "Growth-Adjusted EV/Sales"
            ]
        ]
        .notna()
        .any(axis=1)
    )

    # Must have at least one metric
    # in EVERY pillar
    work["Data_Flag"] = np.where(
        work["Growth_Available"]
        & work["Quality_Available"]
        & work["Valuation_Available"],
        "OK",
        "EXCLUDE"
    )

    scoring = work[
        work["Data_Flag"] == "OK"
    ].copy()

    scoring_universes_pre_gics[year] = scoring

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    print("Eligible universe:", len(work))

    print(
        "Missing both Growth metrics:",
        (~work["Growth_Available"]).sum()
    )

    print(
        "Missing both Quality metrics:",
        (~work["Quality_Available"]).sum()
    )

    print(
        "Missing both Valuation metrics:",
        (~work["Valuation_Available"]).sum()
    )

    print(
        "Scoring universe before GICS check:",
        len(scoring)
    )

    print(
        "Scoring names with missing GICS:",
        scoring["GICS Sector"].isna().sum()
    )

    print("\nGICS sector counts:")
    print(
        scoring["GICS Sector"]
        .value_counts(dropna=False)
        .sort_index()
    )


<a id="section-7"></a>

## 7. Historical Sector Normalization

Historical GICS mappings reconstruct the sector structure at each cohort date, including the 2014 structure and pre-2023 classification reversions. These classifications define the historical peer groups used for scoring.

Validation covers missing classifications, affected companies, sector counts, and small sectors. Numerical sector standardization follows the full-universe transformations in Section 8.


In [ ]:
# --------------------------------------------------
# 2014 historical GICS repair candidates
# --------------------------------------------------

s2014 = scoring_universes_pre_gics[2014].copy()

missing_gics_2014 = s2014[
    s2014["GICS Sector"].isna()
][
    [
        "Ticker",
        "Short Name",
        "Company_Market_Cap_Rank"
    ]
].sort_values("Company_Market_Cap_Rank")

comm_services_2014 = s2014[
    s2014["GICS Sector"] == "Communication Services"
][
    [
        "Ticker",
        "Short Name",
        "GICS Sector",
        "Company_Market_Cap_Rank"
    ]
].sort_values("Company_Market_Cap_Rank")

print("2014 missing GICS:", len(missing_gics_2014))
display(missing_gics_2014)

print("\n2014 current Communication Services names:", len(comm_services_2014))
display(comm_services_2014)


In [ ]:
# --------------------------------------------------
# 2014 historical GICS reconstruction - Step 1
#
# Reverses:
# - 2016 Real Estate sector creation
# - 2018 Communication Services restructuring
# - fills 15 missing historical classifications
# --------------------------------------------------

gics_2014 = scoring_universes_pre_gics[2014].copy()

# Preserve Bloomberg's current classification
gics_2014["Historical GICS Sector"] = (
    gics_2014["GICS Sector"]
)

# --------------------------------------------------
# Explicit historical mappings
# --------------------------------------------------

gics_map_2014 = private_overrides["gics_map_2014"]


# Apply explicit mappings
mapped_mask = gics_2014["Ticker"].isin(
    gics_map_2014
)

gics_2014.loc[
    mapped_mask,
    "Historical GICS Sector"
] = (
    gics_2014.loc[mapped_mask, "Ticker"]
    .map(gics_map_2014)
)


# --------------------------------------------------
# Real Estate did not exist as a standalone
# GICS sector in 2014.
#
# Pre-2016 Real Estate belonged to Financials.
# --------------------------------------------------

real_estate_mask = (
    gics_2014["GICS Sector"] == "Real Estate"
)

gics_2014.loc[
    real_estate_mask,
    "Historical GICS Sector"
] = "Financials"


# Save working version
historical_gics_working = {
    2014: gics_2014
}


# --------------------------------------------------
# Validation
# --------------------------------------------------

print("Remaining missing historical GICS:",
      gics_2014["Historical GICS Sector"]
      .isna()
      .sum())

print(
    "Remaining Communication Services in 2014:",
    (
        gics_2014["Historical GICS Sector"]
        == "Communication Services"
    ).sum()
)

print(
    "Remaining Real Estate in 2014:",
    (
        gics_2014["Historical GICS Sector"]
        == "Real Estate"
    ).sum()
)

print(
    "Rows whose sector changed/fixed:",
    (
        gics_2014["Historical GICS Sector"]
        .fillna("MISSING")
        !=
        gics_2014["GICS Sector"]
        .fillna("MISSING")
    ).sum()
)

print("\n2014 reconstructed sector counts:")
print(
    gics_2014[
        "Historical GICS Sector"
    ]
    .value_counts(dropna=False)
    .sort_index()
)


In [ ]:
# --------------------------------------------------
# Find companies affected by 2023 GICS sector changes
# that appear in our historical scoring universes
# --------------------------------------------------

pre_2023_sector_map = private_overrides["pre_2023_sector_map"]


for year in [2014, 2019, 2022]:

    if year == 2014:
        work = gics_2014.copy()
        sector_col = "Historical GICS Sector"
    else:
        work = scoring_universes_pre_gics[year].copy()
        sector_col = "GICS Sector"

    work["Base_Ticker"] = (
        work["Ticker"]
        .astype(str)
        .str.split()
        .str[0]
    )

    hits = work[
        work["Base_Ticker"].isin(pre_2023_sector_map)
    ].copy()

    hits["Required_Pre_2023_Sector"] = (
        hits["Base_Ticker"]
        .map(pre_2023_sector_map)
    )

    hits["Needs_Change"] = (
        hits[sector_col]
        != hits["Required_Pre_2023_Sector"]
    )

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    if len(hits) == 0:
        print("No affected companies found.")
    else:
        display(
            hits[
                [
                    "Ticker",
                    "Short Name",
                    sector_col,
                    "Required_Pre_2023_Sector",
                    "Needs_Change"
                ]
            ].sort_values("Ticker")
        )


In [ ]:
# --------------------------------------------------
# Apply confirmed pre-2023 GICS reversions
# and identify remaining 2019 missing sectors
# --------------------------------------------------

historical_gics_final_working = {}

for year in [2014, 2019, 2022]:

    # Start from our repaired 2014 version,
    # otherwise start from Bloomberg GICS
    if year == 2014:
        work = gics_2014.copy()
        sector_col = "Historical GICS Sector"

    else:
        work = scoring_universes_pre_gics[year].copy()

        work["Historical GICS Sector"] = (
            work["GICS Sector"]
        )

        sector_col = "Historical GICS Sector"

    # Base ticker
    work["Base_Ticker"] = (
        work["Ticker"]
        .astype(str)
        .str.split()
        .str[0]
    )

    # Apply pre-2023 sector mapping
    affected = work[
        work["Base_Ticker"].isin(
            pre_2023_sector_map
        )
    ].copy()

    for idx in affected.index:

        ticker = work.loc[idx, "Base_Ticker"]

        required_sector = (
            pre_2023_sector_map[ticker]
        )

        work.loc[
            idx,
            "Historical GICS Sector"
        ] = required_sector

    historical_gics_final_working[year] = work

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    changed = work[
        work["GICS Sector"].fillna("MISSING")
        !=
        work["Historical GICS Sector"].fillna("MISSING")
    ]

    print(
        "Total rows differing from Bloomberg current GICS:",
        len(changed)
    )

    print(
        "Remaining missing historical GICS:",
        work["Historical GICS Sector"]
        .isna()
        .sum()
    )


# --------------------------------------------------
# Show remaining 2019 missing classifications
# --------------------------------------------------

missing_2019 = (
    historical_gics_final_working[2019]
    [
        historical_gics_final_working[2019][
            "Historical GICS Sector"
        ].isna()
    ]
    [
        [
            "Ticker",
            "Short Name",
            "Company_Market_Cap_Rank"
        ]
    ]
    .sort_values(
        "Company_Market_Cap_Rank"
    )
)

print("\n2019 missing historical GICS:")
display(missing_2019)


In [ ]:
# --------------------------------------------------
# Fill final 2019 missing historical GICS sectors
# and validate all three cohorts
# --------------------------------------------------

missing_2019_map = private_overrides["missing_2019_map"]


# Apply mappings to 2019
work_2019 = historical_gics_final_working[2019].copy()

mask = work_2019["Ticker"].isin(
    missing_2019_map
)

work_2019.loc[
    mask,
    "Historical GICS Sector"
] = (
    work_2019.loc[mask, "Ticker"]
    .map(missing_2019_map)
)

historical_gics_final_working[2019] = work_2019


# --------------------------------------------------
# Final validation
# --------------------------------------------------

for year, df in historical_gics_final_working.items():

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    print(
        "Scoring universe:",
        len(df)
    )

    print(
        "Missing historical GICS:",
        df["Historical GICS Sector"]
        .isna()
        .sum()
    )

    print("\nHistorical sector counts:")

    print(
        df["Historical GICS Sector"]
        .value_counts()
        .sort_index()
    )


In [ ]:
# Final compact GICS validation

for year, df in historical_gics_final_working.items():

    counts = df["Historical GICS Sector"].value_counts()

    print(
        f"{year} | "
        f"N={len(df)} | "
        f"Missing={df['Historical GICS Sector'].isna().sum()} | "
        f"Sectors={len(counts)} | "
        f"Smallest sector={counts.min()}"
    )


<a id="section-8"></a>

## 8. Factor Transformation & Scoring

Within each cohort, raw factors are winsorized at their nonmissing 1st and 99th percentiles, then percentile-ranked across the full scoring universe. These ranks are standardized within historical GICS sectors using population standard deviation (`ddof=0`). Valid observations in groups with one observation or zero dispersion receive neutral z-scores.

Valuation z-scores are reversed so lower valuations score better. Available component scores are averaged within each pillar. The composite is **0.50 × Growth + 0.30 × Quality + 0.20 × Valuation**. Descending ranks use `method="first"`, which resolves tied scores by row order. Coverage checks identify missing pillar or composite scores.


In [ ]:
# --------------------------------------------------
# STEP 1: Winsorize raw factors at 1st / 99th pct
# STEP 2: Percentile-rank across FULL scoring universe
#
# Do NOT sector z-score yet.
# --------------------------------------------------

factor_metrics = {
    "Revenue Growth Acceleration": "Rev_Accel",
    "3M BF12M EPS Revision": "EPS_Revision",
    "ROIC": "ROIC",
    "FCF MARGIN": "FCF_Margin",
    "BF12M PEG Clean": "PEG",
    "Growth-Adjusted EV/Sales": "GA_EV_Sales"
}

transformed_universes = {}

for year, df in historical_gics_final_working.items():

    work = df.copy()

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    for raw_col, short_name in factor_metrics.items():

        # ---------------------------------
        # 1% / 99% winsorization thresholds
        # calculated only from nonmissing values
        # ---------------------------------

        valid = work[raw_col].dropna()

        lower = valid.quantile(0.01)
        upper = valid.quantile(0.99)

        winsor_col = f"{short_name}_Winsor"

        work[winsor_col] = (
            work[raw_col]
            .clip(
                lower=lower,
                upper=upper
            )
        )

        # ---------------------------------
        # Rank-transform across full cohort
        #
        # Higher raw value = higher percentile
        # Valuation direction will be reversed
        # AFTER sector z-scoring.
        # ---------------------------------

        rank_col = f"{short_name}_Rank"

        work[rank_col] = (
            work[winsor_col]
            .rank(
                method="average",
                pct=True,
                ascending=True
            )
        )

        print(
            f"{short_name:15s} | "
            f"N={valid.count():3d} | "
            f"P01={lower: .6f} | "
            f"P99={upper: .6f} | "
            f"Rank min={work[rank_col].min():.4f} | "
            f"Rank max={work[rank_col].max():.4f}"
        )

    transformed_universes[year] = work


In [ ]:
# --------------------------------------------------
# STEP 3: Sector-neutral z-scores
#
# Input = full-universe percentile ranks
# Standardize WITHIN Historical GICS Sector
# --------------------------------------------------

def safe_sector_zscore(series):

    valid = series.dropna()

    # Preserve all-missing groups
    if len(valid) == 0:
        return pd.Series(
            np.nan,
            index=series.index
        )

    std = valid.std(ddof=0)

    # If only one valid observation or no dispersion,
    # give valid observations a neutral z-score of 0
    if len(valid) < 2 or std == 0:
        result = pd.Series(
            np.nan,
            index=series.index
        )
        result.loc[valid.index] = 0.0
        return result

    return (
        (series - valid.mean())
        / std
    )


zscore_universes = {}

for year, df in transformed_universes.items():

    work = df.copy()

    rank_cols = {
        "Rev_Accel_Rank": "Rev_Accel_Z",
        "EPS_Revision_Rank": "EPS_Revision_Z",
        "ROIC_Rank": "ROIC_Z",
        "FCF_Margin_Rank": "FCF_Margin_Z",
        "PEG_Rank": "PEG_Z",
        "GA_EV_Sales_Rank": "GA_EV_Sales_Z"
    }

    for rank_col, z_col in rank_cols.items():

        work[z_col] = (
            work
            .groupby(
                "Historical GICS Sector",
                group_keys=False
            )[rank_col]
            .transform(safe_sector_zscore)
        )

    zscore_universes[year] = work

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    for z_col in rank_cols.values():

        print(
            f"{z_col:18s} | "
            f"N={work[z_col].notna().sum():3d} | "
            f"Mean={work[z_col].mean(): .6f} | "
            f"Std={work[z_col].std(ddof=0): .6f}"
        )


In [ ]:
# --------------------------------------------------
# STEP 4: Build pillar scores
# STEP 5: Build final 50 / 30 / 20 composite
#
# Growth      = 50%
# Quality     = 30%
# Valuation   = 20%
#
# Within each pillar:
# - both metrics available -> equal weight
# - one metric missing -> remaining metric carries pillar
# --------------------------------------------------

scored_universes = {}

for year, df in zscore_universes.items():

    work = df.copy()

    # ---------------------------------
    # Reverse valuation:
    # lower PEG / GA EV-Sales = better
    # ---------------------------------

    work["PEG_Z_Good"] = -work["PEG_Z"]

    work["GA_EV_Sales_Z_Good"] = (
        -work["GA_EV_Sales_Z"]
    )

    # ---------------------------------
    # Pillars
    #
    # pandas mean(skipna=True) implements
    # our frozen missing-data rule:
    #
    # 2 available -> average the two
    # 1 available -> that metric carries pillar
    # ---------------------------------

    work["Growth_Pillar"] = (
        work[
            [
                "Rev_Accel_Z",
                "EPS_Revision_Z"
            ]
        ]
        .mean(axis=1, skipna=True)
    )

    work["Quality_Pillar"] = (
        work[
            [
                "ROIC_Z",
                "FCF_Margin_Z"
            ]
        ]
        .mean(axis=1, skipna=True)
    )

    work["Valuation_Pillar"] = (
        work[
            [
                "PEG_Z_Good",
                "GA_EV_Sales_Z_Good"
            ]
        ]
        .mean(axis=1, skipna=True)
    )

    # ---------------------------------
    # Final composite
    # ---------------------------------

    work["Composite_Score"] = (
          0.50 * work["Growth_Pillar"]
        + 0.30 * work["Quality_Pillar"]
        + 0.20 * work["Valuation_Pillar"]
    )

    # Rank 1 = strongest composite
    work["Model_Rank"] = (
        work["Composite_Score"]
        .rank(
            method="first",
            ascending=False
        )
        .astype(int)
    )

    work = (
        work
        .sort_values("Model_Rank")
        .reset_index(drop=True)
    )

    scored_universes[year] = work

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    print("Scored companies:", len(work))

    print(
        "Missing Growth Pillar:",
        work["Growth_Pillar"].isna().sum()
    )

    print(
        "Missing Quality Pillar:",
        work["Quality_Pillar"].isna().sum()
    )

    print(
        "Missing Valuation Pillar:",
        work["Valuation_Pillar"].isna().sum()
    )

    print(
        "Missing Composite:",
        work["Composite_Score"].isna().sum()
    )

    print(
        "Composite mean:",
        round(work["Composite_Score"].mean(), 6)
    )

    print(
        "Composite std:",
        round(
            work["Composite_Score"].std(ddof=0),
            6
        )
    )


<a id="section-9"></a>

## 9. Portfolio Formation

Each cohort's portfolio contains the highest-ranked 25 companies; the lowest-ranked 25 form the Bottom 25 comparison. Cutoff checks identify ties at both selection boundaries.

Portfolios start equally weighted and are held for 36 months without monthly rebalancing.


In [ ]:
# --------------------------------------------------
# Freeze Top 25 / Bottom 25 selections
# and validate cutoff ties
# --------------------------------------------------

cohort_selections = {}

for year, df in scored_universes.items():

    work = (
        df
        .sort_values("Model_Rank")
        .reset_index(drop=True)
    )

    top25 = work.head(25).copy()
    bottom25 = work.tail(25).copy()

    top25["Portfolio"] = "Top 25"
    bottom25["Portfolio"] = "Bottom 25"

    cohort_selections[year] = {
        "Top25": top25,
        "Bottom25": bottom25
    }

    # Cutoff scores
    rank25_score = work.iloc[24]["Composite_Score"]
    rank26_score = work.iloc[25]["Composite_Score"]

    bottom25_start = len(work) - 25

    bottom25_score = work.iloc[bottom25_start]["Composite_Score"]
    next_above_bottom_score = work.iloc[bottom25_start - 1]["Composite_Score"]

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    print("Top 25 cutoff:")
    print(
        f"Rank 25 = {rank25_score:.6f} | "
        f"Rank 26 = {rank26_score:.6f} | "
        f"Exact tie = {rank25_score == rank26_score}"
    )

    print("\nBottom 25 cutoff:")
    print(
        f"Rank {bottom25_start} = "
        f"{next_above_bottom_score:.6f} | "
        f"Rank {bottom25_start + 1} = "
        f"{bottom25_score:.6f} | "
        f"Exact tie = "
        f"{next_above_bottom_score == bottom25_score}"
    )

    print("\nTOP 25")
    display(
        top25[
            [
                "Model_Rank",
                "Ticker",
                "Short Name",
                "Historical GICS Sector",
                "Composite_Score"
            ]
        ]
    )

    print("\nBOTTOM 25")
    display(
        bottom25[
            [
                "Model_Rank",
                "Ticker",
                "Short Name",
                "Historical GICS Sector",
                "Composite_Score"
            ]
        ]
    )


<a id="section-10"></a>

## 10. 36-Month Return Analysis

Bloomberg monthly-return exports are parsed into ticker-by-36-month matrices. Layout, coverage, and alias-history checks assess the return inputs. Each company uses the canonical ticker selected during deduplication, without choosing aliases based on subsequent performance.

Monthly total returns are compounded for each constituent. Averaging constituent wealth paths implements equal initial weights with buy-and-hold weight drift. **36-month cumulative total return** is final portfolio wealth minus one. The same function calculates supplementary CAGR and the risk statistics described in Section 13.


In [ ]:
# --------------------------------------------------
# Parse all Bloomberg Monthly Returns sheets
# into clean Ticker x 36-month matrices
# --------------------------------------------------

return_windows = {
    2014: pd.date_range(
        start="2015-01-31",
        end="2017-12-31",
        freq="ME"
    ),

    2019: pd.date_range(
        start="2020-01-31",
        end="2022-12-31",
        freq="ME"
    ),

    2022: pd.date_range(
        start="2023-01-31",
        end="2025-12-31",
        freq="ME"
    )
}

block_starts = [2, 14, 26, 38, 50, 62]

monthly_return_matrices = {}

for year, file in files.items():

    raw = pd.read_excel(
        file,
        sheet_name="Monthly Returns",
        header=None
    )

    dates = return_windows[year]

    all_blocks = []

    for block_num, id_col in enumerate(
        block_starts,
        start=1
    ):

        value_col = id_col + 1

        block = raw.iloc[
            1:,                # remove BQL header row
            [id_col, value_col]
        ].copy()

        block.columns = [
            "Ticker",
            "Monthly_Return"
        ]

        # Remove completely blank rows
        block = block.dropna(
            how="all"
        ).reset_index(drop=True)

        # Numeric cleanup
        block["Monthly_Return"] = pd.to_numeric(
            block["Monthly_Return"],
            errors="coerce"
        )

        print(
            f"{year} Block {block_num} | "
            f"Rows={len(block)} | "
            f"Unique tickers={block['Ticker'].nunique()} | "
            f"Numeric returns={block['Monthly_Return'].notna().sum()}"
        )

        # Validate expected Bloomberg layout
        if len(block) != 18000:
            print(
                f"WARNING: expected 18,000 rows, "
                f"found {len(block)}"
            )

        ticker_counts = (
            block["Ticker"]
            .value_counts()
        )

        if not (ticker_counts == 36).all():
            print(
                "WARNING: some tickers do not "
                "have exactly 36 rows"
            )

        # Add month sequence:
        # each ticker occupies 36 consecutive rows
        block["Month_Number"] = (
            block
            .groupby("Ticker", sort=False)
            .cumcount()
        )

        block["Date"] = block[
            "Month_Number"
        ].map(
            dict(enumerate(dates))
        )

        all_blocks.append(block)

    # Combine all 6 x 500-security blocks
    long_returns = pd.concat(
        all_blocks,
        ignore_index=True
    )

    # Wide format:
    # one row per Bloomberg ticker,
    # one column per month
    wide_returns = (
        long_returns
        .pivot(
            index="Ticker",
            columns="Date",
            values="Monthly_Return"
        )
        .sort_index()
    )

    monthly_return_matrices[year] = wide_returns

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    print(
        "Final matrix shape:",
        wide_returns.shape
    )

    print(
        "Unique tickers:",
        wide_returns.index.nunique()
    )

    print(
        "Months:",
        len(wide_returns.columns)
    )

    print(
        "Missing monthly returns:",
        int(wide_returns.isna().sum().sum())
    )

    print(
        "First month:",
        wide_returns.columns.min()
    )

    print(
        "Last month:",
        wide_returns.columns.max()
    )

    print()


In [ ]:
# --------------------------------------------------
# Validate return matching for scored companies
# and compare multi-alias return histories
# --------------------------------------------------

return_match_summary = {}

for year, scored in scored_universes.items():

    returns = monthly_return_matrices[year]

    canonical_missing = []
    alias_checks = []

    for _, row in scored.iterrows():

        canonical = row["Ticker"]

        aliases = [
            x.strip()
            for x in str(row["Economic_Aliases"]).split(" | ")
        ]

        # -----------------------------
        # Canonical ticker coverage
        # -----------------------------
        if canonical not in returns.index:
            canonical_missing.append(canonical)

        # -----------------------------
        # Compare aliases when >1 exists
        # -----------------------------
        matched_aliases = [
            a for a in aliases
            if a in returns.index
        ]

        if len(matched_aliases) > 1:

            base = returns.loc[
                matched_aliases[0]
            ].astype(float)

            max_diff = 0.0

            for alias in matched_aliases[1:]:

                other = returns.loc[
                    alias
                ].astype(float)

                diff = (
                    base - other
                ).abs().max()

                max_diff = max(
                    max_diff,
                    float(diff)
                )

            alias_checks.append({
                "Canonical": canonical,
                "Aliases": " | ".join(matched_aliases),
                "Alias_Count": len(matched_aliases),
                "Max_Monthly_Return_Diff": max_diff
            })

    alias_checks = pd.DataFrame(alias_checks)

    return_match_summary[year] = alias_checks

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    print(
        "Scored companies:",
        len(scored)
    )

    print(
        "Canonical tickers missing from returns:",
        len(canonical_missing)
    )

    print(
        "Multi-alias companies checked:",
        len(alias_checks)
    )

    if len(alias_checks) > 0:

        print(
            "Alias groups with non-identical returns:",
            (
                alias_checks[
                    "Max_Monthly_Return_Diff"
                ] > 1e-10
            ).sum()
        )

        print(
            "Largest alias monthly-return difference:",
            alias_checks[
                "Max_Monthly_Return_Diff"
            ].max()
        )

        # Only display problematic groups
        problems = alias_checks[
            alias_checks[
                "Max_Monthly_Return_Diff"
            ] > 1e-10
        ]

        if len(problems) > 0:
            display(problems)


In [ ]:
# --------------------------------------------------
# Diagnose non-identical alias return histories
#
# Shows:
# - first month aliases diverge
# - number of differing months
# - 36M compounded return of each alias
# - number of zero-return months
# --------------------------------------------------

alias_return_diagnostics = {}

for year, checks in return_match_summary.items():

    returns = monthly_return_matrices[year]

    problems = checks[
        checks["Max_Monthly_Return_Diff"] > 1e-10
    ].copy()

    rows = []

    for _, row in problems.iterrows():

        canonical = row["Canonical"]

        aliases = [
            x.strip()
            for x in row["Aliases"].split(" | ")
        ]

        base = returns.loc[canonical].astype(float)

        for alias in aliases:

            if alias == canonical:
                continue

            other = returns.loc[alias].astype(float)

            diff = (base - other).abs()

            differing = diff > 1e-10

            if differing.any():
                first_diff = diff[differing].index[0]
                diff_months = int(differing.sum())
            else:
                first_diff = None
                diff_months = 0

            canonical_36m = (
                (1 + base).prod() - 1
            )

            alias_36m = (
                (1 + other).prod() - 1
            )

            rows.append({
                "Canonical": canonical,
                "Alias": alias,
                "First_Diff_Month": first_diff,
                "Different_Months": diff_months,
                "Canonical_36M_Return": canonical_36m,
                "Alias_36M_Return": alias_36m,
                "Canonical_Zero_Months": int(
                    np.isclose(base, 0).sum()
                ),
                "Alias_Zero_Months": int(
                    np.isclose(other, 0).sum()
                )
            })

    diag = pd.DataFrame(rows)

    alias_return_diagnostics[year] = diag

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    if len(diag) == 0:
        print("No problematic alias groups.")
    else:
        display(
            diag.sort_values(
                ["Canonical", "Alias"]
            )
        )


In [ ]:
# --------------------------------------------------
# Build final company-level 36-month return panels
#
# Frozen return rule:
# Use the canonical ticker chosen during company
# deduplication. Do NOT select aliases based on
# realized forward performance.
# --------------------------------------------------

company_return_panels = {}

for year, scored in scored_universes.items():

    returns = monthly_return_matrices[year]

    # Canonical ticker order from scored universe
    tickers = scored["Ticker"].tolist()

    # Pull exactly those return histories
    panel = returns.loc[tickers].copy()

    # Keep model rank / identity alongside returns
    metadata = (
        scored[
            [
                "Ticker",
                "Short Name",
                "Model_Rank",
                "Composite_Score",
                "Historical GICS Sector",
                "Economic_Aliases"
            ]
        ]
        .set_index("Ticker")
    )

    # Compound 36 monthly total returns
    total_36m = (
        (1 + panel)
        .prod(axis=1)
        - 1
    )

    metadata["Return_36M"] = total_36m

    company_return_panels[year] = {
        "Monthly": panel,
        "Metadata": metadata
    }

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    print(
        "Companies:",
        len(panel)
    )

    print(
        "Months:",
        panel.shape[1]
    )

    print(
        "Missing monthly returns:",
        int(panel.isna().sum().sum())
    )

    print(
        "Missing 36M returns:",
        metadata["Return_36M"].isna().sum()
    )

    print(
        "Min 36M return:",
        f"{metadata['Return_36M'].min():.2%}"
    )

    print(
        "Median 36M return:",
        f"{metadata['Return_36M'].median():.2%}"
    )

    print(
        "Max 36M return:",
        f"{metadata['Return_36M'].max():.2%}"
    )


In [ ]:
# --------------------------------------------------
# Top 25 vs Bottom 25
#
# Portfolio rule:
# - Equal weight at formation
# - Buy and hold for 36 months
# - No monthly rebalancing
# --------------------------------------------------

portfolio_results = {}
portfolio_paths = {}

def buy_and_hold_portfolio(monthly_returns):

    # Each stock begins with $1 of wealth
    constituent_wealth = (
        1 + monthly_returns
    ).cumprod(axis=1)

    # Equal initial weights:
    # portfolio wealth = average constituent wealth
    portfolio_wealth = (
        constituent_wealth.mean(axis=0)
    )

    # Derive monthly portfolio returns from wealth path
    portfolio_monthly_return = (
        portfolio_wealth.pct_change()
    )

    portfolio_monthly_return.iloc[0] = (
        portfolio_wealth.iloc[0] - 1
    )

    total_return = (
        portfolio_wealth.iloc[-1] - 1
    )

    cagr = (
        (1 + total_return) ** (1 / 3)
        - 1
    )

    annualized_vol = (
        portfolio_monthly_return.std(ddof=1)
        * np.sqrt(12)
    )

    # Include initial wealth = 1 for drawdown
    wealth_with_start = pd.concat(
        [
            pd.Series(
                [1.0],
                index=[
                    portfolio_wealth.index[0]
                    - pd.offsets.MonthEnd(1)
                ]
            ),
            portfolio_wealth
        ]
    )

    drawdown = (
        wealth_with_start
        / wealth_with_start.cummax()
        - 1
    )

    max_drawdown = drawdown.min()

    return {
        "Monthly_Return": portfolio_monthly_return,
        "Wealth": portfolio_wealth,
        "Total_Return_36M": total_return,
        "CAGR": cagr,
        "Annualized_Volatility": annualized_vol,
        "Max_Drawdown": max_drawdown
    }


for year in [2014, 2019, 2022]:

    panel = company_return_panels[year]["Monthly"]

    top_tickers = (
        cohort_selections[year]["Top25"]["Ticker"]
        .tolist()
    )

    bottom_tickers = (
        cohort_selections[year]["Bottom25"]["Ticker"]
        .tolist()
    )

    top = buy_and_hold_portfolio(
        panel.loc[top_tickers]
    )

    bottom = buy_and_hold_portfolio(
        panel.loc[bottom_tickers]
    )

    portfolio_paths[year] = {
        "Top25": top,
        "Bottom25": bottom
    }

    result = pd.DataFrame(
        {
            "Top 25": [
                top["Total_Return_36M"],
                top["CAGR"],
                top["Annualized_Volatility"],
                top["Max_Drawdown"]
            ],

            "Bottom 25": [
                bottom["Total_Return_36M"],
                bottom["CAGR"],
                bottom["Annualized_Volatility"],
                bottom["Max_Drawdown"]
            ]
        },
        index=[
            "36-Month Cumulative Total Return",
            "CAGR",
            "Annualized Volatility",
            "Max Drawdown"
        ]
    )

    portfolio_results[year] = result

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    display(
        result.style.format("{:.2%}")
    )

    print(
        "Top minus Bottom 36M cumulative total return spread:",
        f"{top['Total_Return_36M'] - bottom['Total_Return_36M']:.2%}"
    )

<a id="section-11"></a>

## 11. Random Portfolio Benchmark

For **each cohort**, 100 separate random portfolios are generated from the **same eligible mid-cap scoring universe** used by the factor model. Each portfolio contains **25 distinct companies selected without replacement**, is **equally weighted at formation**, and is **held for 36 months without monthly rebalancing**. Companies may recur across separate simulations.

A **36-month cumulative total return** is calculated for each portfolio. **Random Portfolio Mean** means the arithmetic mean of those **100 separate portfolio-level 36-month cumulative total returns**. The comparison therefore represents **100 random 25-stock portfolios**.

The Top 25 is compared with the random-portfolio mean, median, 10th–90th percentile range, and percentage of random portfolios beaten. The last measure counts only random portfolios with a strictly lower 36-month cumulative total return than the Top 25.

In [ ]:
# --------------------------------------------------
# 100 random equal-weight 25-stock portfolios
#
# Comparison universe = same scoring universe
# Portfolio rule = same as Top 25:
# equal weight at formation, buy-and-hold 36 months
# --------------------------------------------------

random_portfolio_results = {}

for year in [2014, 2019, 2022]:

    panel = company_return_panels[year]["Monthly"]

    all_tickers = panel.index.to_numpy()

    # Reproducible random generator
    rng = np.random.default_rng(
        seed=42 + year
    )

    random_rows = []

    for simulation in range(1, 101):

        selected = rng.choice(
            all_tickers,
            size=25,
            replace=False
        )

        result = buy_and_hold_portfolio(
            panel.loc[selected]
        )

        random_rows.append({
            "Simulation": simulation,
            "Total_Return_36M":
                result["Total_Return_36M"],
            "CAGR":
                result["CAGR"],
            "Annualized_Volatility":
                result["Annualized_Volatility"],
            "Max_Drawdown":
                result["Max_Drawdown"]
        })

    random_df = pd.DataFrame(
        random_rows
    )

    random_portfolio_results[year] = (
        random_df
    )

    top_return = (
        portfolio_paths[year]
        ["Top25"]
        ["Total_Return_36M"]
    )

    random_mean = (
        random_df["Total_Return_36M"]
        .mean()
    )

    random_median = (
        random_df["Total_Return_36M"]
        .median()
    )

    random_p10 = (
        random_df["Total_Return_36M"]
        .quantile(0.10)
    )

    random_p90 = (
        random_df["Total_Return_36M"]
        .quantile(0.90)
    )

    beats_count = (
        top_return
        >
        random_df["Total_Return_36M"]
    ).sum()

    percentile = (
        beats_count / 100
    )

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    print(
        "Top 25 36-month cumulative total return:",
        f"{top_return:.2%}"
    )

    print(
        "Mean of 100 Random 25-Stock Portfolios (36M cumulative total return):",
        f"{random_mean:.2%}"
    )

    print(
        "Random portfolio median 36M cumulative total return:",
        f"{random_median:.2%}"
    )

    print(
        "Random portfolio 36M cumulative total return, 10th–90th percentile:",
        f"{random_p10:.2%} to {random_p90:.2%}"
    )

    print(
        "Top 25 beat random portfolios:",
        f"{beats_count} / 100"
    )

    print(
        "Percentage of random portfolios beaten (strict outperformance):",
        f"{percentile:.0%}"
    )

<a id="section-12"></a>

## 12. S&P MidCap 400 Benchmark

The `2014 Benchmark`, `2019 Benchmark`, and `2022 Benchmark` sheets provide monthly returns for the **S&P MidCap 400 (MID Index)**. Each cohort is checked for 36 observations and evaluated with the same wealth-path function.

The primary spread is Top 25 cumulative 36-month return minus the index's cumulative 36-month return. CAGR comparisons are supplementary.


In [ ]:
# --------------------------------------------------
# Parse S&P MidCap 400 benchmark (MID Index)
# and compare against Top 25
# --------------------------------------------------

benchmark_results = {}
benchmark_paths = {}

benchmark_sheets = {
    2014: "2014 Benchmark",
    2019: "2019 Benchmark",
    2022: "2022 Benchmark"
}

for year, sheet in benchmark_sheets.items():

    raw = pd.read_excel(
        benchmark_file,
        sheet_name=sheet,
        header=None
    )

    # Remove Bloomberg header row
    benchmark_monthly = pd.to_numeric(
        raw.iloc[1:, 1],
        errors="coerce"
    ).reset_index(drop=True)

    dates = return_windows[year]

    # Validation
    if len(benchmark_monthly) != 36:
        print(
            f"WARNING {year}: expected 36 months, "
            f"found {len(benchmark_monthly)}"
        )

    # One-row matrix so we can use the same
    # buy_and_hold_portfolio() function
    benchmark_panel = pd.DataFrame(
        [benchmark_monthly.to_numpy()],
        index=["MID Index"],
        columns=dates
    )

    benchmark = buy_and_hold_portfolio(
        benchmark_panel
    )

    benchmark_paths[year] = benchmark

    top = portfolio_paths[year]["Top25"]

    benchmark_results[year] = {
        "Benchmark_36M_Return":
            benchmark["Total_Return_36M"],

        "Benchmark_CAGR":
            benchmark["CAGR"],

        "Benchmark_Volatility":
            benchmark["Annualized_Volatility"],

        "Benchmark_Max_Drawdown":
            benchmark["Max_Drawdown"],

        "Top25_36M_Return":
            top["Total_Return_36M"],

        "Top25_Excess_36M":
            (
                top["Total_Return_36M"]
                - benchmark["Total_Return_36M"]
            ),

        "Top25_Excess_CAGR":
            (
                top["CAGR"]
                - benchmark["CAGR"]
            )
    }

    comparison = pd.DataFrame(
        {
            "Top 25": [
                top["Total_Return_36M"],
                top["CAGR"],
                top["Annualized_Volatility"],
                top["Max_Drawdown"]
            ],

            "S&P MidCap 400": [
                benchmark["Total_Return_36M"],
                benchmark["CAGR"],
                benchmark["Annualized_Volatility"],
                benchmark["Max_Drawdown"]
            ]
        },
        index=[
            "36-Month Cumulative Total Return",
            "CAGR",
            "Annualized Volatility",
            "Max Drawdown"
        ]
    )

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    display(
        comparison.style.format("{:.2%}")
    )

    print(
        "Top 25 vs MID 36M cumulative total return spread:",
        f"{benchmark_results[year]['Top25_Excess_36M']:.2%}"
    )

    print(
        "Top 25 excess CAGR:",
        f"{benchmark_results[year]['Top25_Excess_CAGR']:.2%}"
    )

<a id="section-13"></a>

## 13. Risk Analysis

**Annualized volatility** is the sample standard deviation of monthly portfolio returns (`ddof=1`) multiplied by the square root of 12. **Maximum drawdown** is the minimum wealth-to-running-peak ratio minus one, including initial wealth of 1.

The shared portfolio function applies these measures to the Top 25, Bottom 25, random portfolios, and index. The table below reports portfolio risk. Drawdowns use month-end observations and do not capture intramonth extremes.


In [ ]:
for risk_year in [2014, 2019, 2022]:
    print(f"{risk_year} cohort: portfolio risk")
    display(portfolio_results[risk_year].loc[
        ["Annualized Volatility", "Max Drawdown"]
    ].style.format("{:.2%}"))


<a id="section-14"></a>

## 14. Return Statistical Testing

The one-sided empirical return test compares the Top 25 with the sampled distribution of 100 random peer portfolios from the same eligible scoring universe:

**(1 + number of random portfolios with return >= Top 25 return) / (100 + 1)**

All returns in this comparison are **36-month cumulative total returns**. Ties are included in the empirical p-value. In contrast, the **percentage of random portfolios beaten** counts strict outperformance: Top 25 return must exceed the random portfolio's return.

With only 100 simulations, the smallest attainable p-value is 1/101, approximately 0.0099. The test describes performance relative to these sampled peer portfolios; it does not establish persistent alpha. Statistical conclusions require caution given only 100 simulations per cohort and three historical cohorts.

In [ ]:
# One-sided empirical test of 36-month cumulative total return.
return_stat_rows = []
for year in [2014, 2019, 2022]:
    top_return = (
        portfolio_paths[year]
        ["Top25"]
        ["Total_Return_36M"]
    )

    random_returns = (
        random_portfolio_results[year]
        ["Total_Return_36M"]
    )

    return_empirical_p = (
        1
        +
        (
            random_returns
            >= top_return
        ).sum()
    ) / (
        len(random_returns) + 1
    )

    return_stat_rows.append({
        "Cohort": year,
        "Top 25 36M Cumulative Total Return": top_return,
        "Return Empirical p-value": return_empirical_p,
        "Percentage of Random Portfolios Beaten": (top_return > random_returns).mean(),
    })
return_stats = pd.DataFrame(return_stat_rows).set_index("Cohort")
display(return_stats.style.format({
    "Top 25 36M Cumulative Total Return": "{:.2%}",
    "Return Empirical p-value": "{:.4f}",
    "Percentage of Random Portfolios Beaten": "{:.0%}",
}))

<a id="section-15"></a>

## 15. Final Results

The primary results table reports **36-month cumulative total returns** and their differences, expressed in percentage points. **Mean of 100 Random 25-Stock Portfolios** is the arithmetic mean of the 100 separate portfolio-level 36-month cumulative total returns for that cohort. The percentage beaten uses strict outperformance; the empirical p-value includes ties. Volatility is annualized, and maximum drawdown uses monthly wealth observations.

In [ ]:
# --------------------------------------------------
# Build primary investment-performance results table
# --------------------------------------------------

return_rows = []

for year in [2014, 2019, 2022]:

    random_returns = (
        random_portfolio_results[year]
        ["Total_Return_36M"]
    )

    top = portfolio_paths[year]["Top25"]
    bottom = portfolio_paths[year]["Bottom25"]
    bench = benchmark_paths[year]

    return_rows.append({

        "Cohort": year,

        # Universe
        "Scoring Universe":
            len(scored_universes[year]),

        # Returns
        "Top25 Return":
            top["Total_Return_36M"],

        "Bottom25 Return":
            bottom["Total_Return_36M"],

        "MID Return":
            bench["Total_Return_36M"],

        "Top25 vs Bottom Spread":
            (
                top["Total_Return_36M"]
                - bottom["Total_Return_36M"]
            ),

        "Top25 vs MID Spread":
            (
                top["Total_Return_36M"]
                - bench["Total_Return_36M"]
            ),

        "Random Return Mean":
            random_returns.mean(),

        "Top25 Return Beat Random %":
            (
                top["Total_Return_36M"]
                > random_returns
            ).mean(),

        # Risk
        "Top25 Volatility":
            top["Annualized_Volatility"],

        "Top25 Max Drawdown":
            top["Max_Drawdown"]
    })


return_results = pd.DataFrame(
    return_rows
).set_index("Cohort")




In [ ]:
return_results["Return Empirical p-value"] = return_stats["Return Empirical p-value"]
result_labels = {'Scoring Universe': 'Scoring Universe Size', 'Top25 Return': 'Top 25 36M Cumulative Total Return', 'Random Return Mean': 'Mean of 100 Random 25-Stock Portfolios (36M Cumulative Total Return)', 'Top25 Return Beat Random %': 'Percentage of Random Portfolios Beaten', 'Return Empirical p-value': 'Return Empirical p-value', 'MID Return': 'S&P MidCap 400 36M Cumulative Total Return', 'Top25 vs MID Spread': 'Top 25 vs MID Spread (pp)', 'Bottom25 Return': 'Bottom 25 36M Cumulative Total Return', 'Top25 vs Bottom Spread': 'Top 25 vs Bottom 25 Spread (pp)', 'Top25 Volatility': 'Top 25 Annualized Volatility', 'Top25 Max Drawdown': 'Top 25 Maximum Drawdown'}
primary_results = return_results.loc[:, list(result_labels)].rename(columns=result_labels)
result_formats = {column: "{:.2%}" for column in primary_results.columns}
result_formats.update({
    "Scoring Universe Size": "{:,.0f}",
    "Percentage of Random Portfolios Beaten": "{:.0%}",
    "Return Empirical p-value": "{:.4f}",
    "Top 25 vs MID Spread (pp)": lambda value: f"{value * 100:+.2f} pp",
    "Top 25 vs Bottom 25 Spread (pp)": lambda value: f"{value * 100:+.2f} pp",
})
display(primary_results.style.format(result_formats))

<a id="section-16"></a>

## 16. Final Project Charts

Both charts show **36-month cumulative total return** and are generated directly
from the backtest results produced earlier in the notebook. Display labels are
rounded for readability.

The random comparison uses the arithmetic mean of **100 separate 25-stock
portfolios**, each equally weighted at formation and held for 36 months without
monthly rebalancing.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

fig_dir = Path("figures")
fig_dir.mkdir(exist_ok=True)

print("Figure folder ready:", fig_dir.resolve())

In [ ]:
# ----------------------------------------
# Chart 1: 36-Month Cumulative Total Return
# Top 25 vs Bottom 25 vs S&P MidCap 400
# ----------------------------------------

chart1_rows = []

for year in [2014, 2019, 2022]:

    chart1_rows.append({
        "Cohort": str(year),

        "Top 25":
            portfolio_paths[year]["Top25"]["Total_Return_36M"] * 100,

        "Bottom 25":
            portfolio_paths[year]["Bottom25"]["Total_Return_36M"] * 100,

        "S&P MidCap 400":
            benchmark_paths[year]["Total_Return_36M"] * 100
    })

chart1_df = pd.DataFrame(chart1_rows)


x = np.arange(len(chart1_df["Cohort"]))
width = 0.24

fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(
    x - width,
    chart1_df["Top 25"],
    width,
    label="Top 25 Factor-Ranked Portfolio"
)

ax.bar(
    x,
    chart1_df["Bottom 25"],
    width,
    label="Bottom 25 Factor-Ranked Portfolio"
)

ax.bar(
    x + width,
    chart1_df["S&P MidCap 400"],
    width,
    label="S&P MidCap 400"
)

ax.set_title(
    "36-Month Cumulative Total Return by Cohort",
    fontsize=14
)

ax.set_xlabel("Historical Cohort")
ax.set_ylabel("36-Month Cumulative Total Return (%)")

ax.set_xticks(x)
ax.set_xticklabels(chart1_df["Cohort"])

ax.legend()
ax.axhline(0, linewidth=1)

for i, col in enumerate(
    ["Top 25", "Bottom 25", "S&P MidCap 400"]
):
    offset = [-width, 0, width][i]

    for j, value in enumerate(chart1_df[col]):

        ax.text(
            j + offset,
            value + 1,
            f"{value:.1f}%",
            ha="center",
            fontsize=9
        )

plt.tight_layout()

plt.savefig(
    fig_dir / "chart1_36m_return_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ----------------------------------------
# Chart 2: Top 25 vs Mean of
# 100 Random 25-Stock Portfolios
# ----------------------------------------

chart2_rows = []

for year in [2014, 2019, 2022]:

    top_return = (
        portfolio_paths[year]["Top25"]["Total_Return_36M"]
        * 100
    )

    random_mean = (
        random_portfolio_results[year]["Total_Return_36M"].mean()
        * 100
    )

    chart2_rows.append({
        "Cohort": str(year),
        "Top 25": top_return,
        "Mean of 100 Random 25-Stock Portfolios": random_mean
    })

chart2_df = pd.DataFrame(chart2_rows)

x = np.arange(len(chart2_df["Cohort"]))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(
    x - width / 2,
    chart2_df["Top 25"],
    width,
    label="Top 25 Factor-Ranked Portfolio"
)

ax.bar(
    x + width / 2,
    chart2_df["Mean of 100 Random 25-Stock Portfolios"],
    width,
    label="Mean of 100 Random 25-Stock Portfolios"
)

ax.set_title(
    "Top 25 vs Mean of 100 Random Mid-Cap Portfolios",
    fontsize=14
)

ax.set_xlabel("Historical Cohort")
ax.set_ylabel("36-Month Cumulative Total Return (%)")

ax.set_xticks(x)
ax.set_xticklabels(chart2_df["Cohort"])

ax.legend()
ax.axhline(0, linewidth=1)

for i, col in enumerate([
    "Top 25",
    "Mean of 100 Random 25-Stock Portfolios"
]):

    offset = [-width / 2, width / 2][i]

    for j, value in enumerate(chart2_df[col]):

        ax.text(
            j + offset,
            value + 1,
            f"{value:.1f}%",
            ha="center",
            fontsize=9
        )

plt.tight_layout()

plt.savefig(
    fig_dir / "chart2_top25_vs_random_mean.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


<a id="section-17"></a>

## 17. Conclusion

The model’s Top 25 portfolio outperformed comparable mid-cap benchmarks in two of the three historical cohorts, although performance varied materially across periods. The **2019 cohort** delivered a **58.40% 36-month cumulative total return**, compared with **7.74%** for the mean of 100 random 25-stock portfolios, **5.87%** for the S&P MidCap 400, and **48.72%** for the Bottom 25. It provides the strongest evidence: the Top 25 beat all 100 sampled random peer portfolios, with an empirical return p-value of approximately **0.0099**.

The **2022 cohort** returned **82.80%** over 36 months, versus **65.28%** for the random-portfolio mean, **52.87%** for the S&P MidCap 400, and **46.00%** for the Bottom 25. This was economically meaningful relative outperformance, but the Top 25 beat **73%** of random portfolios and its empirical p-value was **0.2772**. Its return was not statistically unusual within the sampled random-portfolio distribution.

The **2014 cohort** is an important failure case. Its **13.43%** 36-month cumulative total return fell below the random-portfolio mean of **25.77%**, the S&P MidCap 400 return of **33.75%**, and the Bottom 25 return of **29.25%**. The Top 25 beat just **16%** of random portfolios, with an empirical p-value of **0.8416**. This material underperformance limits any claim of consistent stock-selection ability.

Higher Top 25 volatility in some cohorts means returns should be assessed alongside annualized volatility and maximum drawdown. Concentration in 25 companies, historical data and identity assumptions, and only three historical cohorts also limit interpretation.

The evidence suggests that a model emphasizing accelerating fundamentals, business quality, and growth-adjusted valuation may help identify subsequent mid-cap outperformers in some environments. It does not establish a persistent or universally reliable source of alpha or support broad market generalization. The study evaluates the full **50% Growth / 30% Quality / 20% Valuation** composite; it does not establish that revenue growth acceleration alone causes or predicts outperformance.

<a id="section-18"></a>

## 18. Appendix — Exploratory Large-Cap Graduation Analysis

This exploratory secondary test examines whether a company enters the monthly Top 500 U.S. economic-company market-cap rankings at least once during its 36-month evaluation window. It is not part of the primary investment thesis. Monthly snapshot preparation, historical FX conversion, company reconstruction, and identity validation support the test.

The appendix requires `Ticker_ID_Map_Working.xlsx` (`data` sheet), 108 monthly workbooks in `Snapshots/`, and Yahoo Finance access for historical monthly FX. Its inputs supplement the private Bloomberg workbooks used in the main analysis.

Two-sided Fisher exact tests compare Top 25 graduation with the rest of the scoring universe, by cohort and pooled. A separate random-portfolio graduation benchmark provides an exploratory comparison. Possible recurring companies and the small number of cohorts limit inference.

### Secondary / exploratory: monthly snapshot preparation

Monthly snapshots supply market-cap rankings. Exchange-code, company-ID, and company-name matching audits support economic-company reconstruction.


In [ ]:
snapshot_files = sorted(snapshot_dir.glob("*.xlsx"))
print("Snapshot files found:", len(snapshot_files))


In [ ]:
# --------------------------------------------------
# Parse all 108 monthly snapshots
# and audit ticker suffixes near Top 500
# --------------------------------------------------


snapshot_frames = []

for file in snapshot_files:

    # ---------------------------------
    # Extract YYYY-MM from filename
    # Handles both:
    # TOP500_2015-01.xlsx
    # TOP500_2025_03.xlsx
    # ---------------------------------

    match = re.search(
        r"(\d{4})[-_](\d{2})",
        file.stem
    )

    if match is None:
        print("Could not parse date:", file.name)
        continue

    year = int(match.group(1))
    month = int(match.group(2))

    snapshot_date = pd.Timestamp(
        year=year,
        month=month,
        day=1
    ) + pd.offsets.MonthEnd(0)

    # ---------------------------------
    # Read Bloomberg data
    # Actual securities begin at row 4
    # ---------------------------------

    df = pd.read_excel(
        file,
        sheet_name="Sheet1",
        header=None,
        skiprows=4,
        names=[
            "Ticker",
            "Short Name",
            "Market Cap Local"
        ]
    )

    # Keep actual equity rows only
    df = df[
        df["Ticker"]
        .astype(str)
        .str.contains("Equity", na=False)
    ].copy()

    df["Market Cap Local"] = pd.to_numeric(
        df["Market Cap Local"],
        errors="coerce"
    )

    df["Snapshot_Date"] = snapshot_date

    # Bloomberg exchange suffix:
    df["Exchange_Code"] = (
        df["Ticker"]
        .astype(str)
        .str.split()
        .str[-2]
    )

    # Raw local-currency rank ONLY for diagnosis
    df = (
        df
        .sort_values(
            "Market Cap Local",
            ascending=False
        )
        .reset_index(drop=True)
    )

    df["Raw_Local_Rank"] = (
        np.arange(1, len(df) + 1)
    )

    snapshot_frames.append(df)


snapshot_long = pd.concat(
    snapshot_frames,
    ignore_index=True
)

print("Snapshots parsed:",
      snapshot_long["Snapshot_Date"].nunique())

print("Total security rows:",
      len(snapshot_long))

print("Missing market caps:",
      snapshot_long["Market Cap Local"].isna().sum())


# --------------------------------------------------
# Audit exchange codes near the graduation cutoff
# --------------------------------------------------

near_cutoff = snapshot_long[
    snapshot_long["Raw_Local_Rank"] <= 650
].copy()

exchange_summary = (
    near_cutoff
    .groupby("Exchange_Code")
    .agg(
        Rows=("Ticker", "size"),
        Unique_Tickers=("Ticker", "nunique"),
        Months=("Snapshot_Date", "nunique")
    )
    .sort_values(
        "Rows",
        ascending=False
    )
)

print("\nExchange codes within raw Top 650:")
display(exchange_summary)


# Show all non-US records near cutoff
non_us_near_cutoff = near_cutoff[
    near_cutoff["Exchange_Code"] != "US"
][
    [
        "Snapshot_Date",
        "Raw_Local_Rank",
        "Ticker",
        "Short Name",
        "Market Cap Local",
        "Exchange_Code"
    ]
].copy()

print(
    "\nNon-US security rows within raw Top 650:",
    len(non_us_near_cutoff)
)

print(
    "Unique non-US tickers:",
    non_us_near_cutoff["Ticker"].nunique()
)

display(
    non_us_near_cutoff
    .drop_duplicates("Ticker")
    .sort_values(
        ["Exchange_Code", "Ticker"]
    )
)


In [ ]:
# --------------------------------------------------
# Join the 28 non-US snapshot tickers to BB Company IDs
# and look for same-ID U.S. ticker counterparts
# --------------------------------------------------

ticker_map = pd.read_excel(
    ticker_map_file,
    sheet_name="data"
)

# Keep only useful columns
ticker_map = ticker_map[
    [
        "Ticker",
        "BB Company ID"
    ]
].copy()

ticker_map["BB Company ID"] = pd.to_numeric(
    ticker_map["BB Company ID"],
    errors="coerce"
)

# One row per ticker
ticker_map = (
    ticker_map
    .dropna(subset=["Ticker"])
    .drop_duplicates("Ticker")
    .reset_index(drop=True)
)

print("Unique tickers in ID map:", len(ticker_map))
print(
    "Missing BB Company IDs:",
    ticker_map["BB Company ID"].isna().sum()
)


# --------------------------------------------------
# Unique non-US tickers seen near raw Top 650
# --------------------------------------------------

non_us_unique = (
    non_us_near_cutoff[
        [
            "Ticker",
            "Short Name",
            "Exchange_Code"
        ]
    ]
    .drop_duplicates("Ticker")
    .merge(
        ticker_map,
        on="Ticker",
        how="left"
    )
)


# --------------------------------------------------
# Find any U.S. ticker sharing the SAME BB Company ID
# --------------------------------------------------

us_map = ticker_map[
    ticker_map["Ticker"]
    .astype(str)
    .str.contains(r" US Equity$", regex=True, na=False)
].copy()

same_id_us = (
    non_us_unique
    .merge(
        us_map,
        on="BB Company ID",
        how="left",
        suffixes=("_NonUS", "_US")
    )
)

same_id_us = same_id_us[
    [
        "Ticker_NonUS",
        "Short Name",
        "Exchange_Code",
        "BB Company ID",
        "Ticker_US"
    ]
].copy()

same_id_us = (
    same_id_us
    .sort_values(
        ["Ticker_NonUS", "Ticker_US"]
    )
    .reset_index(drop=True)
)

print("\nNon-US tickers near Top 650:",
      non_us_unique["Ticker"].nunique())

print(
    "Non-US tickers with missing BB Company ID:",
    non_us_unique["BB Company ID"].isna().sum()
)

print(
    "Non-US tickers with at least one same-ID US ticker:",
    same_id_us[
        "Ticker_US"
    ].notna()
    .groupby(same_id_us["Ticker_NonUS"])
    .any()
    .sum()
)

display(same_id_us)


In [ ]:
# --------------------------------------------------
# Check whether non-US snapshot records have
# likely U.S.-ticker counterparts in the SAME month
# using company-name similarity.
# --------------------------------------------------


def normalize_name(name):
    name = str(name).upper()
    name = re.sub(r"[^A-Z0-9 ]", " ", name)

    remove_words = {
        "INC", "CORP", "CORPORATION", "CO", "COMPANY",
        "LTD", "LIMITED", "PLC", "LLC", "HOLDINGS",
        "HOLDING", "GROUP", "THE", "SA", "NV", "AG",
        "CLASS", "CL", "A", "B"
    }

    words = [
        w for w in name.split()
        if w not in remove_words
    ]

    return " ".join(words)


snapshot_long["Clean_Name"] = (
    snapshot_long["Short Name"]
    .apply(normalize_name)
)

matches = []

for ticker in non_us_unique["Ticker"]:

    rows = snapshot_long[
        snapshot_long["Ticker"] == ticker
    ].copy()

    best_examples = []

    for _, nonus in rows.iterrows():

        same_month_us = snapshot_long[
            (snapshot_long["Snapshot_Date"] == nonus["Snapshot_Date"])
            & (snapshot_long["Exchange_Code"] == "US")
        ].copy()

        if same_month_us.empty:
            continue

        same_month_us["Name_Similarity"] = (
            same_month_us["Clean_Name"]
            .apply(
                lambda x: SequenceMatcher(
                    None,
                    nonus["Clean_Name"],
                    x
                ).ratio()
            )
        )

        best = (
            same_month_us
            .sort_values(
                "Name_Similarity",
                ascending=False
            )
            .iloc[0]
        )

        if best["Name_Similarity"] >= 0.75:

            best_examples.append({
                "NonUS_Ticker": ticker,
                "NonUS_Name": nonus["Short Name"],
                "Date": nonus["Snapshot_Date"],
                "US_Ticker": best["Ticker"],
                "US_Name": best["Short Name"],
                "Name_Similarity": best["Name_Similarity"],
                "NonUS_Local_Cap": nonus["Market Cap Local"],
                "US_Cap": best["Market Cap Local"]
            })

    if best_examples:

        temp = pd.DataFrame(best_examples)

        # keep strongest example for this non-US ticker
        matches.append(
            temp.sort_values(
                "Name_Similarity",
                ascending=False
            ).iloc[0]
        )

name_match_audit = pd.DataFrame(matches)

print(
    "Non-US tickers with likely same-month "
    "U.S.-ticker counterpart:",
    len(name_match_audit)
)

display(
    name_match_audit.sort_values(
        ["Name_Similarity", "NonUS_Ticker"],
        ascending=[False, True]
    )
)


### Secondary / exploratory: historical monthly FX

Yahoo Finance supplies monthly FX rates for converting snapshot market capitalization to USD. Quotes are inverted where necessary and aligned to month-end. The final London snapshot conversion uses GBP rather than pence.

Downloads require network access. Source revisions and availability can affect future reproduction of this exploratory analysis.


In [ ]:
# --------------------------------------------------
# Download monthly FX rates:
# USD value of 1 unit of local currency
# --------------------------------------------------

fx_specs = {

    # Exchange code : (Yahoo symbol, invert?)
    "CN": ("USDCAD=X", True),   # CAD per USD -> USD per CAD
    "HK": ("USDHKD=X", True),   # HKD per USD
    "TT": ("USDTWD=X", True),   # TWD per USD
    "GR": ("EURUSD=X", False),  # USD per EUR
    "KS": ("USDKRW=X", True),   # KRW per USD
    "SS": ("USDSEK=X", True),   # SEK per USD
    "LN": ("GBPUSD=X", False),  # USD per GBP
    "JP": ("USDJPY=X", True),   # JPY per USD
    "NO": ("USDNOK=X", True),   # NOK per USD
}

fx_monthly = {}

for exchange_code, (symbol, invert) in fx_specs.items():

    data = yf.download(
        symbol,
        start="2014-12-01",
        end="2026-01-10",
        auto_adjust=False,
        progress=False
    )

    # Handle either ordinary or MultiIndex output
    if isinstance(data.columns, pd.MultiIndex):
        close = data["Close"].iloc[:, 0]
    else:
        close = data["Close"]

    close = (
        close
        .dropna()
        .resample("ME")
        .last()
    )

    if invert:
        usd_per_local = 1 / close
    else:
        usd_per_local = close

    fx_monthly[exchange_code] = usd_per_local


# Combine into one monthly FX table
fx_table = pd.DataFrame(fx_monthly)

print("FX table shape:", fx_table.shape)
print("First month:", fx_table.index.min())
print("Last month:", fx_table.index.max())

display(fx_table.head())
display(fx_table.tail())


In [ ]:
# --------------------------------------------------
# Final FX setup audit before converting snapshots
# --------------------------------------------------

# London-listed equity values are generally in GBp
# Convert pence -> pounds -> USD
fx_table["LN"] = fx_table["LN"] / 100


# All exchange codes appearing anywhere in snapshots
all_exchange_counts = (
    snapshot_long["Exchange_Code"]
    .value_counts()
    .sort_values(ascending=False)
)

print("All exchange codes:")
display(all_exchange_counts.to_frame("Rows"))


# Codes we currently know how to convert
covered_codes = set(fx_table.columns) | {"US"}

all_codes = set(
    snapshot_long["Exchange_Code"]
    .dropna()
    .unique()
)

unmapped_codes = sorted(
    all_codes - covered_codes
)

print("\nUnmapped exchange codes:")
print(unmapped_codes)


# Restrict FX table to actual snapshot period
fx_table_snapshots = fx_table.loc[
    "2015-01-31":"2025-12-31"
].copy()

print(
    "\nFX months available:",
    len(fx_table_snapshots)
)

print(
    "Missing FX observations by code:"
)

display(
    fx_table_snapshots
    .isna()
    .sum()
    .to_frame("Missing Months")
)


In [ ]:
# --------------------------------------------------
# Add remaining snapshot FX currencies
# --------------------------------------------------

# Italy + Netherlands use EUR
fx_table["IT"] = fx_table["GR"]
fx_table["NA"] = fx_table["GR"]


# Australia: USD per AUD
aud = yf.download(
    "AUDUSD=X",
    start="2014-12-01",
    end="2026-01-10",
    auto_adjust=False,
    progress=False
)

if isinstance(aud.columns, pd.MultiIndex):
    aud = aud["Close"].iloc[:, 0]
else:
    aud = aud["Close"]

fx_table["AU"] = (
    aud
    .dropna()
    .resample("ME")
    .last()
)


# New Zealand: USD per NZD
nzd = yf.download(
    "NZDUSD=X",
    start="2014-12-01",
    end="2026-01-10",
    auto_adjust=False,
    progress=False
)

if isinstance(nzd.columns, pd.MultiIndex):
    nzd = nzd["Close"].iloc[:, 0]
else:
    nzd = nzd["Close"]

fx_table["NZ"] = (
    nzd
    .dropna()
    .resample("ME")
    .last()
)


# Poland: PLN per USD -> invert to USD per PLN
pln = yf.download(
    "USDPLN=X",
    start="2014-12-01",
    end="2026-01-10",
    auto_adjust=False,
    progress=False
)

if isinstance(pln.columns, pd.MultiIndex):
    pln = pln["Close"].iloc[:, 0]
else:
    pln = pln["Close"]

fx_table["PW"] = (
    1 /
    pln
    .dropna()
    .resample("ME")
    .last()
)


# Restrict again to actual snapshot period
fx_table_snapshots = fx_table.loc[
    "2015-01-31":"2025-12-31"
].copy()


# --------------------------------------------------
# Final coverage validation
# --------------------------------------------------

all_codes = set(
    snapshot_long["Exchange_Code"]
    .dropna()
    .unique()
)

covered_codes = (
    set(fx_table_snapshots.columns)
    | {"US"}
)

unmapped_codes = sorted(
    all_codes - covered_codes
)

print(
    "Unmapped exchange codes:",
    unmapped_codes
)

print(
    "FX months:",
    len(fx_table_snapshots)
)

print("\nMissing FX observations:")
display(
    fx_table_snapshots[
        sorted(
            set(fx_table_snapshots.columns)
            & (all_codes - {"US"})
        )
    ]
    .isna()
    .sum()
    .to_frame("Missing Months")
)


In [ ]:
# --------------------------------------------------
# Correct London FX handling
#
# Snapshot Market Cap Local is behaving as GBP,
# not GBp, so restore the original GBP/USD series.
# --------------------------------------------------

fx_table["LN"] = fx_monthly["LN"]

fx_table_snapshots = fx_table.loc[
    "2015-01-31":"2025-12-31"
].copy()

print(
    "2015-01 GBP/USD:",
    fx_table_snapshots.loc[
        "2015-01-31",
        "LN"
    ]
)


In [ ]:
# --------------------------------------------------
# Convert all 324,000 monthly snapshot market caps
# into USD
# --------------------------------------------------

# Long-form FX table
fx_long = (
    fx_table_snapshots
    .stack()
    .rename("FX_to_USD")
    .reset_index()
    .rename(
        columns={
            "Date": "Snapshot_Date",
            "level_1": "Exchange_Code"
        }
    )
)

snapshot_usd = snapshot_long.copy()

# Merge FX for non-US listings
snapshot_usd = snapshot_usd.merge(
    fx_long,
    on=[
        "Snapshot_Date",
        "Exchange_Code"
    ],
    how="left"
)

# U.S. listings already USD
snapshot_usd.loc[
    snapshot_usd["Exchange_Code"] == "US",
    "FX_to_USD"
] = 1.0

# Convert
snapshot_usd["Market Cap USD"] = (
    snapshot_usd["Market Cap Local"]
    * snapshot_usd["FX_to_USD"]
)

print(
    "Rows:",
    len(snapshot_usd)
)

print(
    "Missing FX:",
    snapshot_usd["FX_to_USD"].isna().sum()
)

print(
    "Missing USD market cap:",
    snapshot_usd["Market Cap USD"].isna().sum()
)


# --------------------------------------------------
# Validate the privately configured cross-listing
# --------------------------------------------------

cross_listing = snapshot_usd[
    snapshot_usd["Ticker"].isin(
        [
            private_overrides["cross_listing_check"]["foreign"],
            private_overrides["cross_listing_check"]["us"]
        ]
    )
][
    [
        "Snapshot_Date",
        "Ticker",
        "Market Cap Local",
        "FX_to_USD",
        "Market Cap USD"
    ]
].copy()

cross_listing_compare = (
    cross_listing
    .pivot(
        index="Snapshot_Date",
        columns="Ticker",
        values="Market Cap USD"
    )
    .dropna()
)

cross_listing_compare["Difference_$"] = (
    cross_listing_compare[private_overrides["cross_listing_check"]["foreign"]]
    - cross_listing_compare[private_overrides["cross_listing_check"]["us"]]
)

cross_listing_compare["Difference_%"] = (
    cross_listing_compare["Difference_$"]
    / cross_listing_compare[private_overrides["cross_listing_check"]["us"]]
)

print(
    "\nMonths with both configured listings:",
    len(cross_listing_compare)
)

print(
    "Median absolute difference:",
    f"{cross_listing_compare['Difference_%'].abs().median():.2%}"
)

print(
    "Maximum absolute difference:",
    f"{cross_listing_compare['Difference_%'].abs().max():.2%}"
)

display(
    cross_listing_compare.head(12)
)


### Secondary / exploratory: company reconstruction and identity validation

Recurring duplicate detection and the explicit alias map establish monthly economic-company rankings. Validation checks the Top 500 membership counts before calculating graduation.

A separate company-ID sensitivity audit evaluates additional identity matches without replacing the alias-based graduation results.


In [ ]:
# --------------------------------------------------
# Discover recurring economic-company duplicates
# near the monthly Top 500
#
# Method:
# 1. Exact same USD market cap -> strong duplicate signal
# 2. Same normalized company name + market caps within 5%
#    -> catches cross-currency duplicate cases
#
# We inspect Top 575 as a buffer around the Top 500 cutoff.
# --------------------------------------------------


duplicate_pair_rows = []

for date, month_df in snapshot_usd.groupby("Snapshot_Date"):

    month = (
        month_df
        .sort_values("Market Cap USD", ascending=False)
        .reset_index(drop=True)
    )

    month["Raw_USD_Rank"] = np.arange(
        1,
        len(month) + 1
    )

    check = month[
        month["Raw_USD_Rank"] <= 575
    ].copy()

    # ---------------------------------
    # A) Exact same USD market cap
    # ---------------------------------

    exact_counts = (
        check["Market Cap USD"]
        .value_counts()
    )

    repeated_caps = exact_counts[
        exact_counts > 1
    ].index

    for cap in repeated_caps:

        group = check[
            check["Market Cap USD"] == cap
        ]

        for i, j in combinations(
            group.index,
            2
        ):

            a = group.loc[i]
            b = group.loc[j]

            pair = sorted([
                a["Ticker"],
                b["Ticker"]
            ])

            duplicate_pair_rows.append({
                "Date": date,
                "Ticker_1": pair[0],
                "Ticker_2": pair[1],
                "Reason": "Exact Cap",
                "Rank_1": a["Raw_USD_Rank"],
                "Rank_2": b["Raw_USD_Rank"],
                "Name_1": a["Short Name"],
                "Name_2": b["Short Name"],
                "Cap_Diff_Pct": 0.0
            })

    # ---------------------------------
    # B) Same normalized name
    #    + market caps within 5%
    # ---------------------------------

    for clean_name, group in check.groupby("Clean_Name"):

        if len(group) < 2:
            continue

        for i, j in combinations(
            group.index,
            2
        ):

            a = group.loc[i]
            b = group.loc[j]

            # Skip exact-cap pair already captured
            if a["Market Cap USD"] == b["Market Cap USD"]:
                continue

            cap_ratio = (
                min(
                    a["Market Cap USD"],
                    b["Market Cap USD"]
                )
                /
                max(
                    a["Market Cap USD"],
                    b["Market Cap USD"]
                )
            )

            if cap_ratio >= 0.95:

                pair = sorted([
                    a["Ticker"],
                    b["Ticker"]
                ])

                duplicate_pair_rows.append({
                    "Date": date,
                    "Ticker_1": pair[0],
                    "Ticker_2": pair[1],
                    "Reason": "Same Name + Close Cap",
                    "Rank_1": a["Raw_USD_Rank"],
                    "Rank_2": b["Raw_USD_Rank"],
                    "Name_1": a["Short Name"],
                    "Name_2": b["Short Name"],
                    "Cap_Diff_Pct": 1 - cap_ratio
                })


duplicate_pair_months = pd.DataFrame(
    duplicate_pair_rows
)

# --------------------------------------------------
# Aggregate recurring pairs across 108 months
# --------------------------------------------------

pair_summary = (
    duplicate_pair_months
    .groupby(
        ["Ticker_1", "Ticker_2"],
        as_index=False
    )
    .agg(
        Months_Flagged=("Date", "nunique"),
        First_Month=("Date", "min"),
        Last_Month=("Date", "max"),
        Best_Rank=("Rank_1", lambda x: None),
        Max_Cap_Diff_Pct=("Cap_Diff_Pct", "max")
    )
)

# Recalculate best rank correctly using both sides
best_rank = (
    duplicate_pair_months
    .assign(
        Pair_Best_Rank=lambda x:
            x[["Rank_1", "Rank_2"]].min(axis=1)
    )
    .groupby(
        ["Ticker_1", "Ticker_2"]
    )["Pair_Best_Rank"]
    .min()
)

pair_summary["Best_Rank"] = [
    best_rank.loc[(r["Ticker_1"], r["Ticker_2"])]
    for _, r in pair_summary.iterrows()
]

pair_summary = (
    pair_summary
    .sort_values(
        ["Best_Rank", "Months_Flagged"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print(
    "Unique candidate duplicate pairs:",
    len(pair_summary)
)

print(
    "Pairs ever appearing inside raw Top 500:",
    (pair_summary["Best_Rank"] <= 500).sum()
)

display(
    pair_summary.head(100)
)


In [ ]:
# --------------------------------------------------
# Show representative names + detection reason
# for every candidate duplicate pair
# --------------------------------------------------

pair_details = (
    duplicate_pair_months
    .sort_values("Date")
    .groupby(
        ["Ticker_1", "Ticker_2"],
        as_index=False
    )
    .first()
    [
        [
            "Ticker_1",
            "Name_1",
            "Ticker_2",
            "Name_2",
            "Reason"
        ]
    ]
)

pair_details = (
    pair_summary[
        [
            "Ticker_1",
            "Ticker_2",
            "Months_Flagged",
            "Best_Rank"
        ]
    ]
    .merge(
        pair_details,
        on=["Ticker_1", "Ticker_2"],
        how="left"
    )
    .sort_values("Best_Rank")
    .reset_index(drop=True)
)

display(pair_details)


In [ ]:
# --------------------------------------------------
# Freeze snapshot economic-company alias map
# and reconstruct monthly Top 500 company lists
# --------------------------------------------------

alias_groups = private_overrides["alias_groups"]

# Only configured alias groups are merged; other identities remain separate.


# --------------------------------------------------
# Build alias -> economic key lookup
# --------------------------------------------------

alias_to_economic_key = {}

for economic_key, aliases in alias_groups.items():
    for alias in aliases:
        alias_to_economic_key[alias] = economic_key


monthly_company_rankings = {}
monthly_top500 = {}

validation_rows = []


for date, month_df in snapshot_usd.groupby("Snapshot_Date"):

    month = month_df.copy()

    # Raw USD ranking before dedupe
    month = (
        month
        .sort_values(
            "Market Cap USD",
            ascending=False
        )
        .reset_index(drop=True)
    )

    month["Raw_USD_Rank"] = (
        np.arange(1, len(month) + 1)
    )

    # Economic-company identity
    month["Economic_Key"] = (
        month["Ticker"]
        .map(alias_to_economic_key)
        .fillna(month["Ticker"])
    )

    # Preferred ticker for each alias group.
    # For ordinary companies, the ticker itself is preferred.
    month["Preferred_Record"] = month.apply(
        lambda r:
            r["Ticker"]
            ==
            (
                r["Economic_Key"]
                if r["Economic_Key"] in alias_groups
                and r["Economic_Key"] != private_overrides["snapshot_special_economic_key"]
                else r["Ticker"]
            ),
        axis=1
    )

    # Apply private special-group preferred-record overrides.
    for security, preferred in private_overrides["snapshot_preferred_records"].items():
        month.loc[
            month["Ticker"] == security,
            "Preferred_Record"
        ] = preferred

    # Within each economic company:
    # 1. prefer designated canonical record
    # 2. otherwise use highest USD market-cap record
    company_month = (
        month
        .sort_values(
            [
                "Economic_Key",
                "Preferred_Record",
                "Market Cap USD"
            ],
            ascending=[
                True,
                False,
                False
            ]
        )
        .drop_duplicates(
            subset="Economic_Key",
            keep="first"
        )
        .copy()
    )

    # True economic-company market-cap rank
    company_month = (
        company_month
        .sort_values(
            "Market Cap USD",
            ascending=False
        )
        .reset_index(drop=True)
    )

    company_month["Company_Rank"] = (
        np.arange(1, len(company_month) + 1)
    )

    monthly_company_rankings[date] = company_month

    top500 = company_month.head(500).copy()

    monthly_top500[date] = set(
        top500["Economic_Key"]
    )

    validation_rows.append({
        "Date": date,
        "Raw_Rows": len(month),
        "Economic_Companies": len(company_month),
        "Rows_Collapsed": len(month) - len(company_month),
        "Top500_Count": len(top500),
        "Top500_Unique_Companies":
            top500["Economic_Key"].nunique(),
        "Deepest_Raw_Rank_In_Top500":
            int(top500["Raw_USD_Rank"].max()),
        "500th_Company_Market_Cap":
            top500.iloc[-1]["Market Cap USD"]
    })


top500_validation = pd.DataFrame(
    validation_rows
)

print(
    "Months reconstructed:",
    len(top500_validation)
)

print(
    "Months with exactly 500 unique companies:",
    (
        top500_validation[
            "Top500_Unique_Companies"
        ] == 500
    ).sum(),
    "/",
    len(top500_validation)
)

print(
    "Minimum rows collapsed in a month:",
    top500_validation["Rows_Collapsed"].min()
)

print(
    "Maximum rows collapsed in a month:",
    top500_validation["Rows_Collapsed"].max()
)

print(
    "Deepest raw security rank needed "
    "to reach 500 economic companies:",
    top500_validation[
        "Deepest_Raw_Rank_In_Top500"
    ].max()
)

print("\nFirst 12 months:")
display(
    top500_validation.head(12)
)


In [ ]:
# --------------------------------------------------
# SECONDARY OUTCOME:
# Large-cap graduation within following 36 months
#
# Graduation = economic company appears in reconstructed
# monthly Top 500 at least once during its cohort window.
# --------------------------------------------------

graduation_results = {}
graduation_summary = {}

cohort_months = {
    2014: pd.date_range(
        "2015-01-31",
        "2017-12-31",
        freq="ME"
    ),
    2019: pd.date_range(
        "2020-01-31",
        "2022-12-31",
        freq="ME"
    ),
    2022: pd.date_range(
        "2023-01-31",
        "2025-12-31",
        freq="ME"
    )
}


for year, scored in scored_universes.items():

    work = scored.copy()

    rows = []

    for _, row in work.iterrows():

        # ---------------------------------
        # Build every known identity key
        # for this economic company
        # ---------------------------------

        aliases = [
            x.strip()
            for x in str(
                row["Economic_Aliases"]
            ).split(" | ")
            if x.strip()
        ]

        # Always include canonical ticker
        if row["Ticker"] not in aliases:
            aliases.append(row["Ticker"])

        # Convert aliases into the same economic keys
        # used by reconstructed monthly Top 500 lists
        economic_keys = set()

        for alias in aliases:

            key = alias_to_economic_key.get(
                alias,
                alias
            )

            economic_keys.add(key)

        # ---------------------------------
        # Track Top 500 membership
        # ---------------------------------

        months_in_top500 = []

        best_rank = np.nan

        for date in cohort_months[year]:

            top500_keys = monthly_top500[date]

            if any(
                key in top500_keys
                for key in economic_keys
            ):

                months_in_top500.append(date)

            # Also capture best company rank
            ranking = monthly_company_rankings[date]

            matched = ranking[
                ranking["Economic_Key"].isin(
                    economic_keys
                )
            ]

            if len(matched) > 0:

                month_best = (
                    matched["Company_Rank"].min()
                )

                if pd.isna(best_rank):
                    best_rank = month_best
                else:
                    best_rank = min(
                        best_rank,
                        month_best
                    )

        graduated = (
            len(months_in_top500) > 0
        )

        first_graduation = (
            min(months_in_top500)
            if graduated
            else pd.NaT
        )

        rows.append({
            "Ticker": row["Ticker"],
            "Short Name": row["Short Name"],
            "Model_Rank": row["Model_Rank"],
            "Composite_Score":
                row["Composite_Score"],
            "Historical GICS Sector":
                row["Historical GICS Sector"],

            "Graduated_Top500":
                graduated,

            "First_Graduation_Month":
                first_graduation,

            "Months_In_Top500":
                len(months_in_top500),

            "Best_Company_Rank_36M":
                best_rank
        })


    result = pd.DataFrame(rows)

    graduation_results[year] = result


    # ---------------------------------
    # Portfolio groups
    # ---------------------------------

    top25 = result[
        result["Model_Rank"] <= 25
    ]

    bottom25 = result[
        result["Model_Rank"]
        > len(result) - 25
    ]

    all_scored = result


    # ---------------------------------
    # Graduation rates
    # ---------------------------------

    top_rate = (
        top25["Graduated_Top500"].mean()
    )

    bottom_rate = (
        bottom25["Graduated_Top500"].mean()
    )

    universe_rate = (
        all_scored[
            "Graduated_Top500"
        ].mean()
    )


    summary = pd.DataFrame(
        {
            "Companies": [
                len(top25),
                len(bottom25),
                len(all_scored)
            ],

            "Graduated": [
                top25[
                    "Graduated_Top500"
                ].sum(),

                bottom25[
                    "Graduated_Top500"
                ].sum(),

                all_scored[
                    "Graduated_Top500"
                ].sum()
            ],

            "Graduation Rate": [
                top_rate,
                bottom_rate,
                universe_rate
            ]
        },
        index=[
            "Top 25",
            "Bottom 25",
            "Full Scoring Universe"
        ]
    )

    graduation_summary[year] = summary


    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    display(
        summary.style.format(
            {
                "Graduation Rate": "{:.2%}"
            }
        )
    )

    print(
        "Top 25 minus Bottom 25 graduation spread:",
        f"{top_rate - bottom_rate:.2%}"
    )

    print(
        "Top 25 minus Full Universe graduation spread:",
        f"{top_rate - universe_rate:.2%}"
    )

    print("\nTop 25 graduation details:")

    display(
        top25[
            [
                "Model_Rank",
                "Ticker",
                "Short Name",
                "Graduated_Top500",
                "First_Graduation_Month",
                "Months_In_Top500",
                "Best_Company_Rank_36M"
            ]
        ]
    )


In [ ]:
# --------------------------------------------------
# Graduation identity audit
#
# Compare:
# 1. Existing alias/economic-key matching
# 2. BB Company ID matching
#
# Goal: detect ticker/entity changes that our
# alias-only method may have missed.
# --------------------------------------------------

# Clean ticker -> BB Company ID map
snapshot_id_map = ticker_map[
    ["Ticker", "BB Company ID"]
].copy()

snapshot_id_map["BB Company ID"] = pd.to_numeric(
    snapshot_id_map["BB Company ID"],
    errors="coerce"
)

snapshot_id_map = (
    snapshot_id_map
    .drop_duplicates("Ticker")
)


# Add BB Company ID to reconstructed monthly rankings
monthly_rankings_with_id = {}

for date, ranking in monthly_company_rankings.items():

    temp = ranking.copy()

    temp = temp.merge(
        snapshot_id_map,
        on="Ticker",
        how="left"
    )

    monthly_rankings_with_id[date] = temp


identity_audit = {}

for year, scored in scored_universes.items():

    audit_rows = []

    for _, row in scored.iterrows():

        canonical = row["Ticker"]

        company_id = pd.to_numeric(
            pd.Series(
                [row["BB Company ID"]]
            ),
            errors="coerce"
        ).iloc[0]

        aliases = [
            x.strip()
            for x in str(
                row["Economic_Aliases"]
            ).split(" | ")
            if x.strip()
        ]

        if canonical not in aliases:
            aliases.append(canonical)

        economic_keys = {
            alias_to_economic_key.get(alias, alias)
            for alias in aliases
        }

        alias_months = []
        id_months = []
        union_months = []

        best_alias_rank = np.nan
        best_union_rank = np.nan

        for date in cohort_months[year]:

            ranking = monthly_rankings_with_id[date]

            # -------------------------
            # Alias/economic-key match
            # -------------------------

            alias_match = ranking[
                ranking["Economic_Key"].isin(
                    economic_keys
                )
            ]

            alias_found = len(alias_match) > 0

            if alias_found:

                alias_months.append(date)

                rank = alias_match[
                    "Company_Rank"
                ].min()

                best_alias_rank = (
                    rank
                    if pd.isna(best_alias_rank)
                    else min(best_alias_rank, rank)
                )

            # -------------------------
            # BB Company ID match
            # -------------------------

            if pd.notna(company_id):

                id_match = ranking[
                    ranking["BB Company ID"]
                    == company_id
                ]

            else:
                id_match = ranking.iloc[0:0]

            id_found = len(id_match) > 0

            if id_found:
                id_months.append(date)

            # -------------------------
            # Union identity match
            # -------------------------

            union_match = pd.concat(
                [
                    alias_match,
                    id_match
                ],
                ignore_index=True
            ).drop_duplicates(
                subset="Economic_Key"
            )

            if len(union_match) > 0:

                union_months.append(date)

                rank = union_match[
                    "Company_Rank"
                ].min()

                best_union_rank = (
                    rank
                    if pd.isna(best_union_rank)
                    else min(best_union_rank, rank)
                )

        alias_top500 = (
            pd.notna(best_alias_rank)
            and best_alias_rank <= 500
        )

        union_top500 = (
            pd.notna(best_union_rank)
            and best_union_rank <= 500
        )

        audit_rows.append({
            "Model_Rank": row["Model_Rank"],
            "Ticker": canonical,
            "Short Name": row["Short Name"],
            "BB Company ID": company_id,

            "Alias_Months_Found":
                len(alias_months),

            "ID_Months_Found":
                len(id_months),

            "Union_Months_Found":
                len(set(union_months)),

            "ID_Added_Months":
                len(
                    set(id_months)
                    - set(alias_months)
                ),

            "Best_Alias_Rank":
                best_alias_rank,

            "Best_Union_Rank":
                best_union_rank,

            "Alias_Graduated":
                alias_top500,

            "Union_Graduated":
                union_top500,

            "Graduation_Changed":
                alias_top500 != union_top500
        })

    audit = pd.DataFrame(audit_rows)

    identity_audit[year] = audit

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    print(
        "Companies where Company ID added months:",
        (audit["ID_Added_Months"] > 0).sum()
    )

    print(
        "Companies whose graduation result changed:",
        audit["Graduation_Changed"].sum()
    )

    print(
        "Companies never found by either method:",
        (audit["Union_Months_Found"] == 0).sum()
    )

    print("\nAffected Top 25 names:")

    display(
        audit[
            (audit["Model_Rank"] <= 25)
            &
            (
                (audit["ID_Added_Months"] > 0)
                |
                audit["Graduation_Changed"]
                |
                (audit["Union_Months_Found"] < 36)
            )
        ][
            [
                "Model_Rank",
                "Ticker",
                "Short Name",
                "Alias_Months_Found",
                "ID_Months_Found",
                "Union_Months_Found",
                "ID_Added_Months",
                "Best_Alias_Rank",
                "Best_Union_Rank",
                "Alias_Graduated",
                "Union_Graduated",
                "Graduation_Changed"
            ]
        ]
    )


### Exploratory graduation benchmarks and statistical tests

The random graduation benchmark and Fisher exact tests below evaluate the secondary outcome. The comprehensive statistical summary also carries the return p-values for the private research audit trail.

In [ ]:
# --------------------------------------------------
# SECONDARY OUTCOME RANDOM BENCHMARK
#
# Compare Top 25 graduation rate against
# 100 random 25-company portfolios
# from the same scoring universe.
# --------------------------------------------------

random_graduation_results = {}

for year in [2014, 2019, 2022]:

    result = graduation_results[year].copy()

    rng = np.random.default_rng(
        seed=42 + year
    )

    random_rows = []

    for simulation in range(1, 101):

        sample = result.sample(
            n=25,
            replace=False,
            random_state=int(
                rng.integers(
                    0,
                    2_147_483_647
                )
            )
        )

        graduates = int(
            sample["Graduated_Top500"].sum()
        )

        rate = (
            sample["Graduated_Top500"].mean()
        )

        random_rows.append({
            "Simulation": simulation,
            "Graduated": graduates,
            "Graduation_Rate": rate
        })

    random_df = pd.DataFrame(
        random_rows
    )

    random_graduation_results[year] = (
        random_df
    )

    top25 = result[
        result["Model_Rank"] <= 25
    ]

    top_graduates = int(
        top25["Graduated_Top500"].sum()
    )

    top_rate = (
        top25["Graduated_Top500"].mean()
    )

    mean_rate = (
        random_df["Graduation_Rate"].mean()
    )

    median_rate = (
        random_df["Graduation_Rate"].median()
    )

    p10 = (
        random_df["Graduation_Rate"]
        .quantile(0.10)
    )

    p90 = (
        random_df["Graduation_Rate"]
        .quantile(0.90)
    )

    beats_count = (
        top_rate
        >
        random_df["Graduation_Rate"]
    ).sum()

    ties_count = (
        top_rate
        ==
        random_df["Graduation_Rate"]
    ).sum()

    print(f"\n{'=' * 60}")
    print(year)
    print(f"{'=' * 60}")

    print(
        "Top 25 graduates:",
        f"{top_graduates} / 25"
    )

    print(
        "Top 25 graduation rate:",
        f"{top_rate:.2%}"
    )

    print(
        "Random mean graduation rate:",
        f"{mean_rate:.2%}"
    )

    print(
        "Random median graduation rate:",
        f"{median_rate:.2%}"
    )

    print(
        "Random 10th–90th percentile:",
        f"{p10:.2%} to {p90:.2%}"
    )

    print(
        "Top 25 beat random portfolios:",
        f"{beats_count} / 100"
    )

    print(
        "Top 25 tied random portfolios:",
        f"{ties_count} / 100"
    )


In [ ]:
# --------------------------------------------------
# APPENDIX: COMPREHENSIVE STATISTICAL SUMMARY
#
# 1. Graduation:
#    Top 25 vs REST of scoring universe
#    Fisher exact test by cohort + pooled
#
# 2. Random-portfolio empirical p-values
#    for graduation and 36M returns
#
# Interpret each p-value within its stated comparison and study limitations.
# --------------------------------------------------


stat_rows = []

pooled_top_grad = 0
pooled_top_n = 0

pooled_rest_grad = 0
pooled_rest_n = 0


for year in [2014, 2019, 2022]:

    grad = graduation_results[year]

    # ---------------------------------
    # Top 25
    # ---------------------------------

    top = grad[
        grad["Model_Rank"] <= 25
    ]

    rest = grad[
        grad["Model_Rank"] > 25
    ]

    top_grad = int(
        top["Graduated_Top500"].sum()
    )

    top_non = (
        len(top) - top_grad
    )

    rest_grad = int(
        rest["Graduated_Top500"].sum()
    )

    rest_non = (
        len(rest) - rest_grad
    )

    # 2x2 table
    table = [
        [top_grad, top_non],
        [rest_grad, rest_non]
    ]

    odds_ratio, fisher_p = fisher_exact(
        table,
        alternative="two-sided"
    )

    # ---------------------------------
    # Empirical random-basket
    # graduation p-value
    #
    # One-sided:
    # probability random portfolio does
    # at least as well as Top 25
    # ---------------------------------

    top_grad_rate = (
        top["Graduated_Top500"].mean()
    )

    random_grad = (
        random_graduation_results[year]
        ["Graduation_Rate"]
    )

    grad_empirical_p = (
        1
        +
        (
            random_grad
            >= top_grad_rate
        ).sum()
    ) / (
        len(random_grad) + 1
    )

    # ---------------------------------
    # Empirical return p-value
    # ---------------------------------

    top_return = (
        portfolio_paths[year]
        ["Top25"]
        ["Total_Return_36M"]
    )

    random_returns = (
        random_portfolio_results[year]
        ["Total_Return_36M"]
    )

    return_empirical_p = (
        1
        +
        (
            random_returns
            >= top_return
        ).sum()
    ) / (
        len(random_returns) + 1
    )

    # ---------------------------------
    # Save
    # ---------------------------------

    stat_rows.append({
        "Cohort": year,

        "Top25 Graduates":
            top_grad,

        "Top25 Graduation Rate":
            top_grad_rate,

        "Rest Graduation Rate":
            rest_grad / len(rest),

        "Graduation Odds Ratio":
            odds_ratio,

        "Fisher Exact p":
            fisher_p,

        "Graduation Random p":
            grad_empirical_p,

        "Top25 36M Return":
            top_return,

        "Return Random p":
            return_empirical_p
    })

    # pooled totals
    pooled_top_grad += top_grad
    pooled_top_n += len(top)

    pooled_rest_grad += rest_grad
    pooled_rest_n += len(rest)


# --------------------------------------------------
# Pooled Fisher exact test
# --------------------------------------------------

pooled_table = [
    [
        pooled_top_grad,
        pooled_top_n - pooled_top_grad
    ],
    [
        pooled_rest_grad,
        pooled_rest_n - pooled_rest_grad
    ]
]

pooled_odds_ratio, pooled_fisher_p = (
    fisher_exact(
        pooled_table,
        alternative="two-sided"
    )
)


stats_summary = pd.DataFrame(
    stat_rows
).set_index("Cohort")


display(
    stats_summary.style.format(
        {
            "Top25 Graduation Rate": "{:.2%}",
            "Rest Graduation Rate": "{:.2%}",
            "Graduation Odds Ratio": "{:.3f}",
            "Fisher Exact p": "{:.4f}",
            "Graduation Random p": "{:.4f}",
            "Top25 36M Return": "{:.2%}",
            "Return Random p": "{:.4f}"
        }
    )
)


print("\n" + "=" * 60)
print("POOLED GRADUATION TEST")
print("=" * 60)

print(
    "Top 25 graduates:",
    f"{pooled_top_grad} / {pooled_top_n}",
    f"= {pooled_top_grad / pooled_top_n:.2%}"
)

print(
    "Rest-of-universe graduates:",
    f"{pooled_rest_grad} / {pooled_rest_n}",
    f"= {pooled_rest_grad / pooled_rest_n:.2%}"
)

print(
    "Pooled graduation spread:",
    f"{(pooled_top_grad / pooled_top_n) - (pooled_rest_grad / pooled_rest_n):+.2%}"
)

print(
    "Pooled odds ratio:",
    f"{pooled_odds_ratio:.3f}"
)

print(
    "Pooled Fisher exact p-value:",
    f"{pooled_fisher_p:.4f}"
)

### Exploratory graduation finding

Pooled Top 25 graduation was **14.67%**, compared with **14.18%** for the rest of the scoring universe. The pooled two-sided Fisher exact p-value was **0.8669**. This exploratory test did not reveal a meaningful pooled effect. Movement across an arbitrary Top-500 market-cap threshold is less directly relevant to an investor than realized total return.

### Supporting results: secondary / exploratory large-cap graduation

The comprehensive table includes graduation rates and random-graduation comparisons alongside the investment-return results. Graduation measures entry into the monthly Top 500; it is not a substitute for stock-return performance.


In [ ]:
# --------------------------------------------------
# Build master cohort results table
# --------------------------------------------------

master_rows = []

for year in [2014, 2019, 2022]:

    grad = graduation_results[year]

    top_grad = grad[
        grad["Model_Rank"] <= 25
    ]["Graduated_Top500"].mean()

    bottom_grad = grad[
        grad["Model_Rank"] > len(grad) - 25
    ]["Graduated_Top500"].mean()

    universe_grad = (
        grad["Graduated_Top500"].mean()
    )

    random_grad = (
        random_graduation_results[year]
        ["Graduation_Rate"]
    )

    random_returns = (
        random_portfolio_results[year]
        ["Total_Return_36M"]
    )

    top = portfolio_paths[year]["Top25"]
    bottom = portfolio_paths[year]["Bottom25"]
    bench = benchmark_paths[year]

    master_rows.append({

        "Cohort": year,

        # Universe
        "Scoring Universe":
            len(scored_universes[year]),

        # Graduation
        "Top25 Graduation":
            top_grad,

        "Bottom25 Graduation":
            bottom_grad,

        "Universe Graduation":
            universe_grad,

        "Top25 vs Universe Grad Spread":
            top_grad - universe_grad,

        "Random Graduation Mean":
            random_grad.mean(),

        "Top25 Grad Beat Random %":
            (
                top_grad > random_grad
            ).mean(),

        # Returns
        "Top25 Return":
            top["Total_Return_36M"],

        "Bottom25 Return":
            bottom["Total_Return_36M"],

        "MID Return":
            bench["Total_Return_36M"],

        "Top25 vs Bottom Spread":
            (
                top["Total_Return_36M"]
                - bottom["Total_Return_36M"]
            ),

        "Top25 vs MID Spread":
            (
                top["Total_Return_36M"]
                - bench["Total_Return_36M"]
            ),

        "Random Return Mean":
            random_returns.mean(),

        "Top25 Return Beat Random %":
            (
                top["Total_Return_36M"]
                > random_returns
            ).mean(),

        # Risk
        "Top25 Volatility":
            top["Annualized_Volatility"],

        "Top25 Max Drawdown":
            top["Max_Drawdown"]
    })


master_results = pd.DataFrame(
    master_rows
).set_index("Cohort")




In [ ]:
display(
    master_results.style.format(
        {
            "Top25 Graduation": "{:.2%}",
            "Bottom25 Graduation": "{:.2%}",
            "Universe Graduation": "{:.2%}",
            "Top25 vs Universe Grad Spread": "{:+.2%}",
            "Random Graduation Mean": "{:.2%}",
            "Top25 Grad Beat Random %": "{:.0%}",

            "Top25 Return": "{:.2%}",
            "Bottom25 Return": "{:.2%}",
            "MID Return": "{:.2%}",
            "Top25 vs Bottom Spread": "{:+.2%}",
            "Top25 vs MID Spread": "{:+.2%}",
            "Random Return Mean": "{:.2%}",
            "Top25 Return Beat Random %": "{:.0%}",

            "Top25 Volatility": "{:.2%}",
            "Top25 Max Drawdown": "{:.2%}"
        }
    )
)


### Private research export

The next cell writes `Final_Historical_Research_Results.xlsx`, including aggregate results, statistical tests, company rankings, selections, and graduation outcomes. This detailed workbook is a private research output and is excluded from the public repository.


In [ ]:
# --------------------------------------------------
# Save final research outputs
# --------------------------------------------------

output_file = "Final_Historical_Research_Results.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    # Master results
    master_results.to_excel(
        writer,
        sheet_name="Master Results"
    )

    # Statistical tests
    stats_summary.to_excel(
        writer,
        sheet_name="Statistical Tests"
    )

    # Cohort-specific outputs
    for year in [2014, 2019, 2022]:

        scored_universes[year].to_excel(
            writer,
            sheet_name=f"{year} Rankings",
            index=False
        )

        cohort_selections[year]["Top25"].to_excel(
            writer,
            sheet_name=f"{year} Top25",
            index=False
        )

        cohort_selections[year]["Bottom25"].to_excel(
            writer,
            sheet_name=f"{year} Bottom25",
            index=False
        )

        graduation_results[year].to_excel(
            writer,
            sheet_name=f"{year} Graduation",
            index=False
        )

        random_portfolio_results[year].to_excel(
            writer,
            sheet_name=f"{year} Random Returns",
            index=False
        )

        random_graduation_results[year].to_excel(
            writer,
            sheet_name=f"{year} Random Graduation",
            index=False
        )

print("Saved:", output_file)
